# Análise exploratória — Sobreposição e protocolos v2

Este notebook usa:

- `gold_apuracao_uc`: intervalos, UC, conjunto, regional e tipo de protocolo da UC;
- `gold_interrupcao_tratada`: número do protocolo da UC/interrupção.

Objetivos:

1. Identificar sobreposição de interrupções da mesma UC com `TIPO_PROTOC_JUSTIF_UCI` diferentes.
2. Verificar, no mesmo dia e conjunto, se há UCs com e sem protocolo recuperado.
3. Exportar amostras para investigação manual.

A análise é exploratória e não altera nenhuma tabela gold.

In [1]:
import os
from datetime import datetime
from pathlib import Path

import duckdb
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

ANOMES = os.getenv("ANOMES", "202606")
BASE_DIR = Path("..") if Path.cwd().name.lower() == "notebooks" else Path(".")
DB_PATH = BASE_DIR / "data" / "processed" / f"iqs_adms_processed_{ANOMES}.duckdb"
MARTS_DIR = BASE_DIR / "data" / "marts"
MARTS_DIR.mkdir(parents=True, exist_ok=True)

DATA_INICIO = f"{ANOMES[:4]}-{ANOMES[4:6]}-01"
DATA_FIM_EXCLUSIVA = (pd.Timestamp(DATA_INICIO) + pd.offsets.MonthBegin(1)).strftime("%Y-%m-%d")

print("ANOMES:", ANOMES)
print("DuckDB:", DB_PATH)
print("Período:", DATA_INICIO, "até", DATA_FIM_EXCLUSIVA, "exclusivo")
assert DB_PATH.exists(), f"DuckDB processado não encontrado: {DB_PATH}"

ANOMES: 202606
DuckDB: ..\data\processed\iqs_adms_processed_202606.duckdb
Período: 2026-06-01 até 2026-07-01 exclusivo


In [2]:
con = duckdb.connect(str(DB_PATH), read_only=True)

def tabela_existe(nome_tabela: str) -> bool:
    return con.execute(
        """
        SELECT COUNT(*)
        FROM information_schema.tables
        WHERE table_schema = 'main'
          AND table_name = ?
        """,
        [nome_tabela],
    ).fetchone()[0] > 0

def listar_colunas(nome_tabela: str) -> pd.DataFrame:
    return con.execute(
        """
        SELECT ordinal_position, column_name, data_type
        FROM information_schema.columns
        WHERE table_schema = 'main'
          AND table_name = ?
        ORDER BY ordinal_position
        """,
        [nome_tabela],
    ).fetchdf()

assert tabela_existe("gold_apuracao_uc"), "Tabela gold_apuracao_uc não encontrada. Execute run.bat apuracao_parcial."
assert tabela_existe("gold_interrupcao_tratada"), "Tabela gold_interrupcao_tratada não encontrada. Execute run.bat apuracao_parcial."

listar_colunas("gold_apuracao_uc")

,ordinal_position,column_name,data_type
0,1,REGIONAL,VARCHAR
1,2,NUM_OCORRENCIA_ADMS,VARCHAR
2,3,NUM_SEQ_INTRP,VARCHAR
3,4,NUM_INTRP_UCI,VARCHAR
4,5,NUM_POSTO_UCI,VARCHAR
5,6,NUM_UC_UCI,VARCHAR
6,7,ESTADO_INTRP,VARCHAR
7,8,NUM_MOTIVO_TRAT_DIF_UCI,VARCHAR
8,9,INDIC_SIT_PROCES_INDIC_UCI,VARCHAR
9,10,TIPO_PROTOC_JUSTIF_UCI,VARCHAR


## 1. View enriquecida com protocolo

`gold_apuracao_uc` não possui `NUM_PROTOC_JUSTIF_RESP_UCI`. Por isso o protocolo é recuperado de `gold_interrupcao_tratada`.

Chave usada:

- `NUM_OCORRENCIA_ADMS`
- `NUM_SEQ_INTRP`
- `NUM_INTRP_UCI`
- `NUM_POSTO_UCI`
- `NUM_UC_UCI`

In [3]:
con.execute("""
CREATE OR REPLACE TEMP VIEW base_apuracao_com_protocolo AS
SELECT
    a.REGIONAL,
    a.NUM_OCORRENCIA_ADMS,
    a.NUM_SEQ_INTRP,
    a.NUM_INTRP_UCI,
    a.NUM_POSTO_UCI,
    a.NUM_UC_UCI,
    a.ESTADO_INTRP,
    a.TIPO_PROTOC_JUSTIF_UCI,
    a.NUM_INTRP_INIC_MANOBRA_UCI,
    a.COD_CONJTO_ELET_ANEEL_INTRP,
    a.COD_CAUSA_INTRP,
    a.COD_COMP_INTRP,
    a.COD_TIPO_INTRP,
    a.DATA_HORA_INIC_INTRP,
    a.DATA_HORA_FIM_INTRP,
    a.DTHR_INICIO_INTRP_UC,
    a.DURACAO_HORA,
    a.INTERRUPCAO_LONGA,
    a.INTERRUPCAO_CONTABILIZAVEL,
    a.CI_BRUTO,
    a.CHI_BRUTO,
    a.CI_LIQUIDO,
    a.CHI_LIQUIDO,
    NULLIF(TRIM(CAST(t.NUM_PROTOC_JUSTIF_RESP_UCI AS VARCHAR)), '') AS NUM_PROTOC_JUSTIF_RESP_UCI,
    NULLIF(TRIM(CAST(t.NUM_PROTOC_JUSTIF_RESP_INTRP AS VARCHAR)), '') AS NUM_PROTOC_JUSTIF_RESP_INTRP,
    NULLIF(TRIM(CAST(t.TIPO_PROTOC_JUSTIF_INTRP AS VARCHAR)), '') AS TIPO_PROTOC_JUSTIF_INTRP,
    COALESCE(
        NULLIF(TRIM(CAST(t.NUM_PROTOC_JUSTIF_RESP_UCI AS VARCHAR)), ''),
        NULLIF(TRIM(CAST(t.NUM_PROTOC_JUSTIF_RESP_INTRP AS VARCHAR)), '')
    ) AS PROTOCOLO_RECUPERADO
FROM gold_apuracao_uc a
LEFT JOIN gold_interrupcao_tratada t
  ON TRIM(CAST(t.NUM_OCORRENCIA_ADMS AS VARCHAR)) = TRIM(CAST(a.NUM_OCORRENCIA_ADMS AS VARCHAR))
 AND TRIM(CAST(t.NUM_SEQ_INTRP AS VARCHAR)) = TRIM(CAST(a.NUM_SEQ_INTRP AS VARCHAR))
 AND TRIM(CAST(t.NUM_INTRP_UCI AS VARCHAR)) = TRIM(CAST(a.NUM_INTRP_UCI AS VARCHAR))
 AND TRIM(CAST(t.NUM_POSTO_UCI AS VARCHAR)) = TRIM(CAST(a.NUM_POSTO_UCI AS VARCHAR))
 AND TRIM(CAST(t.NUM_UC_UCI AS VARCHAR)) = TRIM(CAST(a.NUM_UC_UCI AS VARCHAR))
""")

con.execute("SELECT COUNT(*) AS linhas FROM base_apuracao_com_protocolo").fetchdf()

,linhas
0,12262287


## 2. Base exploratória padronizada

In [4]:
con.execute(f"""
CREATE OR REPLACE TEMP VIEW base_eventos_exploratoria AS
SELECT
    ROW_NUMBER() OVER () AS ID_EVENTO,
    NULLIF(TRIM(CAST(NUM_UC_UCI AS VARCHAR)), '') AS UC,
    NULLIF(TRIM(CAST(COD_CONJTO_ELET_ANEEL_INTRP AS VARCHAR)), '') AS CONJUNTO,
    NULLIF(TRIM(CAST(REGIONAL AS VARCHAR)), '') AS REGIONAL,
    NULLIF(TRIM(CAST(NUM_OCORRENCIA_ADMS AS VARCHAR)), '') AS NUM_OCORRENCIA_ADMS,
    NULLIF(TRIM(CAST(NUM_SEQ_INTRP AS VARCHAR)), '') AS NUM_SEQ_INTRP,
    NULLIF(TRIM(CAST(NUM_INTRP_UCI AS VARCHAR)), '') AS NUM_INTRP_UCI,
    NULLIF(TRIM(CAST(NUM_POSTO_UCI AS VARCHAR)), '') AS NUM_POSTO_UCI,
    CAST(DTHR_INICIO_INTRP_UC AS TIMESTAMP) AS INICIO,
    CAST(DATA_HORA_FIM_INTRP AS TIMESTAMP) AS FIM,
    CAST(CAST(DTHR_INICIO_INTRP_UC AS DATE) AS DATE) AS DATA_INTRP,
    NULLIF(TRIM(CAST(TIPO_PROTOC_JUSTIF_UCI AS VARCHAR)), '') AS TIPO_PROTOCOLO_UC,
    NULLIF(TRIM(CAST(TIPO_PROTOC_JUSTIF_INTRP AS VARCHAR)), '') AS TIPO_PROTOCOLO_INTRP,
    NULLIF(TRIM(CAST(NUM_PROTOC_JUSTIF_RESP_UCI AS VARCHAR)), '') AS PROTOCOLO_UC,
    NULLIF(TRIM(CAST(NUM_PROTOC_JUSTIF_RESP_INTRP AS VARCHAR)), '') AS PROTOCOLO_INTRP,
    NULLIF(TRIM(CAST(PROTOCOLO_RECUPERADO AS VARCHAR)), '') AS PROTOCOLO_RECUPERADO,
    CASE WHEN NULLIF(TRIM(CAST(PROTOCOLO_RECUPERADO AS VARCHAR)), '') IS NULL THEN 'N' ELSE 'S' END AS TEM_PROTOCOLO,
    NULLIF(TRIM(CAST(COD_CAUSA_INTRP AS VARCHAR)), '') AS COD_CAUSA_INTRP,
    NULLIF(TRIM(CAST(COD_COMP_INTRP AS VARCHAR)), '') AS COD_COMP_INTRP,
    NULLIF(TRIM(CAST(COD_TIPO_INTRP AS VARCHAR)), '') AS COD_TIPO_INTRP,
    COALESCE(TRY_CAST(DURACAO_HORA AS DOUBLE), 0) AS DURACAO_HORA,
    COALESCE(TRY_CAST(CI_BRUTO AS DOUBLE), 0) AS CI_BRUTO,
    COALESCE(TRY_CAST(CHI_BRUTO AS DOUBLE), 0) AS CHI_BRUTO,
    COALESCE(TRY_CAST(CI_LIQUIDO AS DOUBLE), 0) AS CI_LIQUIDO,
    COALESCE(TRY_CAST(CHI_LIQUIDO AS DOUBLE), 0) AS CHI_LIQUIDO
FROM base_apuracao_com_protocolo
WHERE NULLIF(TRIM(CAST(NUM_UC_UCI AS VARCHAR)), '') IS NOT NULL
  AND CAST(DTHR_INICIO_INTRP_UC AS TIMESTAMP) < TIMESTAMP '{DATA_FIM_EXCLUSIVA}'
  AND CAST(DATA_HORA_FIM_INTRP AS TIMESTAMP) >= TIMESTAMP '{DATA_INICIO}'
  AND CAST(DATA_HORA_FIM_INTRP AS TIMESTAMP) > CAST(DTHR_INICIO_INTRP_UC AS TIMESTAMP)
""")

con.execute("SELECT COUNT(*) AS linhas FROM base_eventos_exploratoria").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,linhas
0,12262287


In [5]:
resumo_base = con.execute("""
SELECT
    COUNT(*) AS LINHAS_EVENTO,
    COUNT(DISTINCT UC) AS UCS,
    COUNT(DISTINCT CONJUNTO) AS CONJUNTOS,
    SUM(CASE WHEN TEM_PROTOCOLO = 'S' THEN 1 ELSE 0 END) AS LINHAS_COM_PROTOCOLO,
    SUM(CASE WHEN TEM_PROTOCOLO = 'N' THEN 1 ELSE 0 END) AS LINHAS_SEM_PROTOCOLO,
    COUNT(DISTINCT TIPO_PROTOCOLO_UC) AS TIPOS_PROTOCOLO_UC,
    COUNT(DISTINCT PROTOCOLO_RECUPERADO) AS PROTOCOLOS_RECUPERADOS
FROM base_eventos_exploratoria
""").fetchdf()
resumo_base

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,LINHAS_EVENTO,UCS,CONJUNTOS,LINHAS_COM_PROTOCOLO,LINHAS_SEM_PROTOCOLO,TIPOS_PROTOCOLO_UC,PROTOCOLOS_RECUPERADOS
0,12262287,2974957,157,3478240.0,8784047.0,3,27


## 3. Sobreposição da mesma UC com `TIPO_PROTOC_JUSTIF_UCI` diferentes

Critério:

- mesma UC;
- intervalos se cruzam;
- `TIPO_PROTOCOLO_UC` diferente.

A duração de sobreposição é a interseção dos intervalos.

In [6]:
con.execute("""
CREATE OR REPLACE TEMP VIEW exploratoria_sobreposicao_tipo_protocolo AS
SELECT
    a.UC,
    a.CONJUNTO,
    a.REGIONAL,
    a.DATA_INTRP,
    a.ID_EVENTO AS ID_EVENTO_A,
    b.ID_EVENTO AS ID_EVENTO_B,
    a.NUM_OCORRENCIA_ADMS AS OCORRENCIA_A,
    b.NUM_OCORRENCIA_ADMS AS OCORRENCIA_B,
    a.NUM_SEQ_INTRP AS INTERRUPCAO_A,
    b.NUM_SEQ_INTRP AS INTERRUPCAO_B,
    a.NUM_INTRP_UCI AS INTRP_UCI_A,
    b.NUM_INTRP_UCI AS INTRP_UCI_B,
    a.TIPO_PROTOCOLO_UC AS TIPO_PROTOCOLO_UC_A,
    b.TIPO_PROTOCOLO_UC AS TIPO_PROTOCOLO_UC_B,
    a.PROTOCOLO_RECUPERADO AS PROTOCOLO_A,
    b.PROTOCOLO_RECUPERADO AS PROTOCOLO_B,
    a.COD_CAUSA_INTRP AS COD_CAUSA_A,
    b.COD_CAUSA_INTRP AS COD_CAUSA_B,
    a.COD_COMP_INTRP AS COD_COMP_A,
    b.COD_COMP_INTRP AS COD_COMP_B,
    a.INICIO AS INICIO_A,
    a.FIM AS FIM_A,
    b.INICIO AS INICIO_B,
    b.FIM AS FIM_B,
    GREATEST(a.INICIO, b.INICIO) AS INICIO_SOBREPOSTO,
    LEAST(a.FIM, b.FIM) AS FIM_SOBREPOSTO,
    DATE_DIFF('second', GREATEST(a.INICIO, b.INICIO), LEAST(a.FIM, b.FIM)) / 3600.0 AS HORAS_SOBREPOSTAS
FROM base_eventos_exploratoria a
JOIN base_eventos_exploratoria b
  ON a.UC = b.UC
 AND a.ID_EVENTO < b.ID_EVENTO
 AND a.INICIO < b.FIM
 AND a.FIM > b.INICIO
 AND COALESCE(a.TIPO_PROTOCOLO_UC, 'NULO') <> COALESCE(b.TIPO_PROTOCOLO_UC, 'NULO')
""")

con.execute("""
SELECT
    COUNT(*) AS PARES_SOBREPOSTOS,
    COUNT(DISTINCT UC) AS UCS_COM_SOBREPOSICAO,
    SUM(HORAS_SOBREPOSTAS) AS HORAS_SOBREPOSTAS,
    MAX(HORAS_SOBREPOSTAS) AS MAIOR_SOBREPOSICAO_H
FROM exploratoria_sobreposicao_tipo_protocolo
""").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,PARES_SOBREPOSTOS,UCS_COM_SOBREPOSICAO,HORAS_SOBREPOSTAS,MAIOR_SOBREPOSICAO_H
0,2438,1238,1288.488333,22.816667


In [13]:
sobreposicao_amostra = con.execute("""
SELECT *
FROM exploratoria_sobreposicao_tipo_protocolo
ORDER BY HORAS_SOBREPOSTAS DESC, UC, DATA_INTRP
--LIMIT 200
""").fetchdf()
sobreposicao_amostra

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,UC,CONJUNTO,REGIONAL,DATA_INTRP,ID_EVENTO_A,ID_EVENTO_B,OCORRENCIA_A,OCORRENCIA_B,INTERRUPCAO_A,INTERRUPCAO_B,...,COD_CAUSA_B,COD_COMP_A,COD_COMP_B,INICIO_A,FIM_A,INICIO_B,FIM_B,INICIO_SOBREPOSTO,FIM_SOBREPOSTO,HORAS_SOBREPOSTAS
0,7356005,15970,LES,2026-06-21,3999591,9828773,1-26447473,1-26450808,261104361611043638,261104346611043609,...,13,42,42,2026-06-21 17:34:00,2026-06-23 12:24:00,2026-06-22 13:11:00,2026-06-23 12:00:00,2026-06-22 13:11:00,2026-06-23 12:00:00,22.816667
1,96824760,14672,NRO,2026-06-11,4582143,8403074,1-26405670,1-26406121,261092952410929566,261092771110929520,...,13,92,35,2026-06-11 23:19:00,2026-06-12 15:36:00,2026-06-12 00:40:00,2026-06-12 15:35:00,2026-06-12 00:40:00,2026-06-12 15:35:00,14.916667
2,32412169,16774,LES,2026-06-12,3114802,7992169,1-26412477,1-26416687,261092987810941744,261094097210941719,...,18,40,88,2026-06-12 15:45:00,2026-06-13 18:23:19,2026-06-13 09:04:00,2026-06-13 18:20:39,2026-06-13 09:04:00,2026-06-13 18:20:39,9.277500
3,76386953,14699,NRT,2026-06-12,1313140,11159120,1-26406037,1-26412002,261093025910930279,261092943110929922,...,82,52,92,2026-06-12 01:37:00,2026-06-12 16:09:00,2026-06-12 07:35:00,2026-06-12 15:46:00,2026-06-12 07:35:00,2026-06-12 15:46:00,8.183333
4,107709155,14647,NRO,2026-06-29,3312077,5364667,1-26492633,1-26504348,261115006311150065,261115790911157952,...,04,26,20,2026-06-29 10:42:00,2026-06-30 16:50:00,2026-06-30 10:08:00,2026-06-30 17:53:00,2026-06-30 10:08:00,2026-06-30 16:50:00,6.700000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2433,99550687,14632,CSL,2026-06-28,8256471,10790782,1-26487997,1-26490026,261112700511127930,261111952711119529,...,82,38,92,2026-06-28 22:22:00,2026-06-29 14:39:55,2026-06-29 08:19:42,2026-06-29 08:19:46,2026-06-29 08:19:42,2026-06-29 08:19:46,0.001111
2434,99596458,15970,LES,2026-06-28,2316790,8600832,1-26488056,1-26488516,261113929211139294,261111688511116886,...,82,92,92,2026-06-28 16:57:00,2026-06-29 16:58:38,2026-06-29 02:00:22,2026-06-29 02:00:26,2026-06-29 02:00:22,2026-06-29 02:00:26,0.001111
2435,99712270,15970,LES,2026-06-28,592378,10132497,1-26488056,1-26488516,261113929211139294,261111688511116886,...,82,92,92,2026-06-28 16:57:00,2026-06-29 16:58:38,2026-06-29 02:00:22,2026-06-29 02:00:26,2026-06-29 02:00:22,2026-06-29 02:00:26,0.001111
2436,99778769,14668,OES,2026-06-27,149524,12058549,1-26481232,1-26486769,261111334311113345,261111290711112908,...,82,26,92,2026-06-27 15:11:00,2026-06-28 19:12:00,2026-06-28 18:43:56,2026-06-28 18:44:00,2026-06-28 18:43:56,2026-06-28 18:44:00,0.001111


## Potencial de ganhos de CHI se reclassificado para dia critico

In [ ]:
potencial_chi_tipo_0 = con.execute("""
WITH pares_sobrepostos AS (
    SELECT
        a.UC,
        a.CONJUNTO,
        a.REGIONAL,
        a.DATA_INTRP,

        a.ID_EVENTO AS ID_EVENTO_A,
        b.ID_EVENTO AS ID_EVENTO_B,

        a.NUM_OCORRENCIA_ADMS AS OCORRENCIA_A,
        b.NUM_OCORRENCIA_ADMS AS OCORRENCIA_B,

        a.NUM_SEQ_INTRP AS INTERRUPCAO_A,
        b.NUM_SEQ_INTRP AS INTERRUPCAO_B,

        a.NUM_INTRP_UCI AS INTRP_UCI_A,
        b.NUM_INTRP_UCI AS INTRP_UCI_B,

        a.TIPO_PROTOCOLO_UC AS TIPO_PROTOCOLO_UC_A,
        b.TIPO_PROTOCOLO_UC AS TIPO_PROTOCOLO_UC_B,

        a.PROTOCOLO_RECUPERADO AS PROTOCOLO_A,
        b.PROTOCOLO_RECUPERADO AS PROTOCOLO_B,

        a.INICIO AS INICIO_A,
        a.FIM AS FIM_A,
        b.INICIO AS INICIO_B,
        b.FIM AS FIM_B,

        GREATEST(a.INICIO, b.INICIO) AS INICIO_SOBREPOSTO,
        LEAST(a.FIM, b.FIM) AS FIM_SOBREPOSTO,

        DATE_DIFF(
            'second',
            GREATEST(a.INICIO, b.INICIO),
            LEAST(a.FIM, b.FIM)
        ) / 3600.0 AS HORAS_SOBREPOSTAS,

        a.CHI_LIQUIDO AS CHI_LIQUIDO_A,
        b.CHI_LIQUIDO AS CHI_LIQUIDO_B,

        CASE
            WHEN a.TIPO_PROTOCOLO_UC = '0'
            THEN a.CHI_LIQUIDO
            ELSE 0
        END AS POTENCIAL_CHI_TIPO_0_A,

        CASE
            WHEN b.TIPO_PROTOCOLO_UC = '0'
            THEN b.CHI_LIQUIDO
            ELSE 0
        END AS POTENCIAL_CHI_TIPO_0_B

    FROM base_eventos_exploratoria a
    JOIN base_eventos_exploratoria b
      ON a.UC = b.UC
     AND a.ID_EVENTO < b.ID_EVENTO
     AND a.INICIO < b.FIM
     AND a.FIM > b.INICIO
     AND COALESCE(a.TIPO_PROTOCOLO_UC, 'NULO') <> COALESCE(b.TIPO_PROTOCOLO_UC, 'NULO')
),
eventos_tipo_0_com_sobreposicao AS (
    SELECT DISTINCT
        UC,
        CONJUNTO,
        REGIONAL,
        DATA_INTRP,
        ID_EVENTO_A AS ID_EVENTO,
        OCORRENCIA_A AS NUM_OCORRENCIA_ADMS,
        INTERRUPCAO_A AS NUM_SEQ_INTRP,
        INTRP_UCI_A AS NUM_INTRP_UCI,
        TIPO_PROTOCOLO_UC_A AS TIPO_PROTOCOLO_UC,
        PROTOCOLO_A AS PROTOCOLO_RECUPERADO,
        INICIO_A AS INICIO,
        FIM_A AS FIM,
        CHI_LIQUIDO_A AS CHI_LIQUIDO,
        POTENCIAL_CHI_TIPO_0_A AS POTENCIAL_CHI_TIPO_0
    FROM pares_sobrepostos
    WHERE TIPO_PROTOCOLO_UC_A = '0'

    UNION

    SELECT DISTINCT
        UC,
        CONJUNTO,
        REGIONAL,
        DATA_INTRP,
        ID_EVENTO_B AS ID_EVENTO,
        OCORRENCIA_B AS NUM_OCORRENCIA_ADMS,
        INTERRUPCAO_B AS NUM_SEQ_INTRP,
        INTRP_UCI_B AS NUM_INTRP_UCI,
        TIPO_PROTOCOLO_UC_B AS TIPO_PROTOCOLO_UC,
        PROTOCOLO_B AS PROTOCOLO_RECUPERADO,
        INICIO_B AS INICIO,
        FIM_B AS FIM,
        CHI_LIQUIDO_B AS CHI_LIQUIDO,
        POTENCIAL_CHI_TIPO_0_B AS POTENCIAL_CHI_TIPO_0
    FROM pares_sobrepostos
    WHERE TIPO_PROTOCOLO_UC_B = '0'
)
SELECT
    UC,
    CONJUNTO,
    REGIONAL,
    DATA_INTRP,
    COUNT(DISTINCT ID_EVENTO) AS QTD_REGISTROS_TIPO_0_SOBREPOSTOS,
    COUNT(DISTINCT NUM_SEQ_INTRP) AS QTD_INTERRUPCOES_TIPO_0,
    SUM(POTENCIAL_CHI_TIPO_0) AS POTENCIAL_CHI_TIPO_0_RECLASSIFICACAO,
    STRING_AGG(DISTINCT NUM_OCORRENCIA_ADMS, ', ') AS OCORRENCIAS,
    STRING_AGG(DISTINCT NUM_SEQ_INTRP, ', ') AS INTERRUPCOES,
    STRING_AGG(DISTINCT PROTOCOLO_RECUPERADO, ', ') FILTER (
        WHERE PROTOCOLO_RECUPERADO IS NOT NULL
    ) AS PROTOCOLOS_ENVOLVIDOS
FROM eventos_tipo_0_com_sobreposicao
GROUP BY
    UC,
    CONJUNTO,
    REGIONAL,
    DATA_INTRP
ORDER BY
    POTENCIAL_CHI_TIPO_0_RECLASSIFICACAO DESC,
    UC,
    DATA_INTRP
""").fetchdf()

potencial_chi_tipo_0

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,UC,CONJUNTO,REGIONAL,DATA_INTRP,QTD_REGISTROS_TIPO_0_SOBREPOSTOS,QTD_INTERRUPCOES_TIPO_0,POTENCIAL_CHI_TIPO_0_RECLASSIFICACAO,OCORRENCIAS,INTERRUPCOES,PROTOCOLOS_ENVOLVIDOS
0,10995374,15576,OES,2026-06-26,1,1,97.871667,1-26473131,261114161611141649,None
1,99440849,14628,OES,2026-06-26,1,1,77.883333,1-26473381,261113598011136238,None
2,60779730,15970,LES,2026-06-21,1,1,76.066667,1-26447678,261105500911056877,None
3,7355920,15970,LES,2026-06-20,1,1,64.483333,1-26444216,261104077111041535,None
4,108728161,15577,OES,2026-06-27,1,1,48.166667,1-26481885,261113703711137068,None
...,...,...,...,...,...,...,...,...,...,...
1491,99900220,15958,CSL,2026-06-29,1,1,0.000000,1-26499782,261114021211140217,None
1492,99957116,15950,LES,2026-06-13,1,1,0.000000,1-26461312,261105106211051067,None
1493,99957132,15950,LES,2026-06-13,1,1,0.000000,1-26461312,261105106211051067,None
1494,99957655,15970,LES,2026-06-12,1,1,0.000000,1-26461312,261105106211051067,None


In [ ]:

potencial_recuperacao = potencial_chi_tipo_0['POTENCIAL_CHI_TIPO_0_RECLASSIFICACAO'].sum()
potencial_recuperacao


np.float64(24442.959444444445)

In [19]:
potencial_chi_tipo_0 = con.execute("""
WITH pares_sobrepostos AS (
    SELECT
        a.UC,
        a.CONJUNTO,
        a.REGIONAL,
        a.DATA_INTRP,

        a.ID_EVENTO AS ID_EVENTO_A,
        b.ID_EVENTO AS ID_EVENTO_B,

        a.NUM_OCORRENCIA_ADMS AS OCORRENCIA_A,
        b.NUM_OCORRENCIA_ADMS AS OCORRENCIA_B,

        a.NUM_SEQ_INTRP AS INTERRUPCAO_A,
        b.NUM_SEQ_INTRP AS INTERRUPCAO_B,

        a.NUM_INTRP_UCI AS INTRP_UCI_A,
        b.NUM_INTRP_UCI AS INTRP_UCI_B,

        a.TIPO_PROTOCOLO_UC AS TIPO_PROTOCOLO_UC_A,
        b.TIPO_PROTOCOLO_UC AS TIPO_PROTOCOLO_UC_B,

        a.PROTOCOLO_RECUPERADO AS PROTOCOLO_A,
        b.PROTOCOLO_RECUPERADO AS PROTOCOLO_B,

        a.INICIO AS INICIO_A,
        a.FIM AS FIM_A,
        b.INICIO AS INICIO_B,
        b.FIM AS FIM_B,

        GREATEST(a.INICIO, b.INICIO) AS INICIO_SOBREPOSTO,
        LEAST(a.FIM, b.FIM) AS FIM_SOBREPOSTO,

        DATE_DIFF(
            'second',
            GREATEST(a.INICIO, b.INICIO),
            LEAST(a.FIM, b.FIM)
        ) / 3600.0 AS HORAS_SOBREPOSTAS,

        a.CHI_LIQUIDO AS CHI_LIQUIDO_A,
        b.CHI_LIQUIDO AS CHI_LIQUIDO_B,

        CASE
            WHEN a.TIPO_PROTOCOLO_UC = '0'
            THEN a.CHI_LIQUIDO
            ELSE 0
        END AS POTENCIAL_CHI_TIPO_0_A,

        CASE
            WHEN b.TIPO_PROTOCOLO_UC = '0'
            THEN b.CHI_LIQUIDO
            ELSE 0
        END AS POTENCIAL_CHI_TIPO_0_B

    FROM base_eventos_exploratoria a
    JOIN base_eventos_exploratoria b
      ON a.UC = b.UC
     AND a.ID_EVENTO < b.ID_EVENTO
     AND a.INICIO < b.FIM
     AND a.FIM > b.INICIO
     AND COALESCE(a.TIPO_PROTOCOLO_UC, 'NULO') <> COALESCE(b.TIPO_PROTOCOLO_UC, 'NULO')
),
eventos_tipo_0_com_sobreposicao AS (
    SELECT DISTINCT
        UC,
        CONJUNTO,
        REGIONAL,
        DATA_INTRP,
        ID_EVENTO_A AS ID_EVENTO,
        OCORRENCIA_A AS NUM_OCORRENCIA_ADMS,
        INTERRUPCAO_A AS NUM_SEQ_INTRP,
        INTRP_UCI_A AS NUM_INTRP_UCI,
        TIPO_PROTOCOLO_UC_A AS TIPO_PROTOCOLO_UC,
        PROTOCOLO_A AS PROTOCOLO_RECUPERADO,
        INICIO_A AS INICIO,
        FIM_A AS FIM,
        CHI_LIQUIDO_A AS CHI_LIQUIDO,
        POTENCIAL_CHI_TIPO_0_A AS POTENCIAL_CHI_TIPO_0
    FROM pares_sobrepostos
    WHERE TIPO_PROTOCOLO_UC_A = '0'

    UNION

    SELECT DISTINCT
        UC,
        CONJUNTO,
        REGIONAL,
        DATA_INTRP,
        ID_EVENTO_B AS ID_EVENTO,
        OCORRENCIA_B AS NUM_OCORRENCIA_ADMS,
        INTERRUPCAO_B AS NUM_SEQ_INTRP,
        INTRP_UCI_B AS NUM_INTRP_UCI,
        TIPO_PROTOCOLO_UC_B AS TIPO_PROTOCOLO_UC,
        PROTOCOLO_B AS PROTOCOLO_RECUPERADO,
        INICIO_B AS INICIO,
        FIM_B AS FIM,
        CHI_LIQUIDO_B AS CHI_LIQUIDO,
        POTENCIAL_CHI_TIPO_0_B AS POTENCIAL_CHI_TIPO_0
    FROM pares_sobrepostos
    WHERE TIPO_PROTOCOLO_UC_B = '0'
),
potencial_por_uc_dia AS (
    SELECT
        UC,
        CONJUNTO,
        REGIONAL,
        DATA_INTRP,
        COUNT(DISTINCT ID_EVENTO) AS QTD_REGISTROS_TIPO_0_SOBREPOSTOS,
        COUNT(DISTINCT NUM_SEQ_INTRP) AS QTD_INTERRUPCOES_TIPO_0,
        SUM(POTENCIAL_CHI_TIPO_0) AS POTENCIAL_CHI_TIPO_0_RECLASSIFICACAO,
        STRING_AGG(DISTINCT NUM_OCORRENCIA_ADMS, ', ') AS OCORRENCIAS,
        STRING_AGG(DISTINCT NUM_SEQ_INTRP, ', ') AS INTERRUPCOES,
        STRING_AGG(DISTINCT PROTOCOLO_RECUPERADO, ', ') FILTER (
            WHERE PROTOCOLO_RECUPERADO IS NOT NULL
        ) AS PROTOCOLOS_ENVOLVIDOS
    FROM eventos_tipo_0_com_sobreposicao
    GROUP BY
        UC,
        CONJUNTO,
        REGIONAL,
        DATA_INTRP
),
base_ressarcimento AS (
    SELECT
        p.*,
        c.FATURADA,
        c.META_DIC,
        c.DIC,
        c.DIC_BASE_COMPENSACAO,
        c.COMP_DIC,
        c.KEI AS KEI1_CONTINUIDADE,
        c.VRC AS VRC_CONTINUIDADE,
        COALESCE(c.VRC, v.VRC) AS VRC_RECUPERADO
    FROM potencial_por_uc_dia p
    LEFT JOIN gold_continuidade_uc c
      ON TRIM(CAST(c.UC AS VARCHAR)) = TRIM(CAST(p.UC AS VARCHAR))
    LEFT JOIN gold_vrc v
      ON TRIM(CAST(v.ISN_UC AS VARCHAR)) = TRIM(CAST(p.UC AS VARCHAR))
)
SELECT
    UC,
    CONJUNTO,
    REGIONAL,
    DATA_INTRP,

    QTD_REGISTROS_TIPO_0_SOBREPOSTOS,
    QTD_INTERRUPCOES_TIPO_0,

    POTENCIAL_CHI_TIPO_0_RECLASSIFICACAO,

    FATURADA,
    META_DIC,
    DIC,
    DIC_BASE_COMPENSACAO,
    COMP_DIC,
    KEI1_CONTINUIDADE,

    VRC_RECUPERADO,

    CASE
        WHEN FATURADA = 'S'
         AND COALESCE(VRC_RECUPERADO, 0) > 0
         AND COALESCE(META_DIC, 0) > 0
         AND COALESCE(POTENCIAL_CHI_TIPO_0_RECLASSIFICACAO, 0) > 0
        THEN
            (
                COALESCE(POTENCIAL_CHI_TIPO_0_RECLASSIFICACAO, 0)
                / META_DIC
            )
            * COALESCE(VRC_RECUPERADO, 0)
            * COALESCE(KEI1_CONTINUIDADE, 0)
        ELSE 0
    END AS RESSARCIMENTO_CHI_TIPO_0_ESTIMADO,

    OCORRENCIAS,
    INTERRUPCOES,
    PROTOCOLOS_ENVOLVIDOS

FROM base_ressarcimento
ORDER BY
    RESSARCIMENTO_CHI_TIPO_0_ESTIMADO DESC,
    POTENCIAL_CHI_TIPO_0_RECLASSIFICACAO DESC,
    UC,
    DATA_INTRP
""").fetchdf()

potencial_chi_tipo_0

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,UC,CONJUNTO,REGIONAL,DATA_INTRP,QTD_REGISTROS_TIPO_0_SOBREPOSTOS,QTD_INTERRUPCOES_TIPO_0,POTENCIAL_CHI_TIPO_0_RECLASSIFICACAO,FATURADA,META_DIC,DIC,DIC_BASE_COMPENSACAO,COMP_DIC,KEI1_CONTINUIDADE,VRC_RECUPERADO,RESSARCIMENTO_CHI_TIPO_0_ESTIMADO,OCORRENCIAS,INTERRUPCOES,PROTOCOLOS_ENVOLVIDOS
0,105540889,16760,OES,2026-06-26,1,1,45.216667,S,13.0,0.000000,0.000000,0.000000,40,41980.86,5.840722e+06,1-26474785,261111290611112937,None
1,77325400,15970,LES,2026-06-28,1,1,24.027222,S,19.0,27.137778,27.137778,66635.854217,40,44812.23,2.266765e+06,1-26488056,261113929211139294,None
2,77325400,15970,LES,2026-06-29,1,1,24.027222,S,19.0,27.137778,27.137778,66635.854217,40,44812.23,2.266765e+06,1-26488056,261113929211139294,None
3,44773439,15575,OES,2026-06-29,1,1,6.933333,S,3.0,6.933333,6.933333,919.033279,40,2419.09,2.236314e+05,1-26489135,261111851511119424,None
4,43152538,14658,OES,2026-06-29,1,1,28.461389,S,20.0,33.779722,33.779722,7180.755455,34,4564.13,2.208325e+05,1-26489388,261111916711144844,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1491,99900220,15958,CSL,2026-06-29,1,1,0.000000,S,20.0,0.000000,0.000000,0.000000,34,10.34,0.000000e+00,1-26499782,261114021211140217,None
1492,99957116,15950,LES,2026-06-13,1,1,0.000000,S,20.0,5.882222,5.882222,0.000000,34,0.00,0.000000e+00,1-26461312,261105106211051067,None
1493,99957132,15950,LES,2026-06-13,1,1,0.000000,S,20.0,5.882222,5.882222,0.000000,34,29.43,0.000000e+00,1-26461312,261105106211051067,None
1494,99957655,15970,LES,2026-06-12,1,1,0.000000,S,20.0,5.882222,5.882222,0.000000,34,14.79,0.000000e+00,1-26461312,261105106211051067,None


In [20]:

potencial_recuperacao_vrc = potencial_chi_tipo_0['VRC_RECUPERADO'].sum()
potencial_recuperacao_vrc

np.float64(233612.75)

In [8]:
sobreposicao_por_uc = con.execute("""
SELECT
    UC,
    CONJUNTO,
    REGIONAL,
    COUNT(*) AS PARES_SOBREPOSTOS,
    SUM(HORAS_SOBREPOSTAS) AS HORAS_SOBREPOSTAS,
    STRING_AGG(DISTINCT COALESCE(TIPO_PROTOCOLO_UC_A, 'NULO') || ' x ' || COALESCE(TIPO_PROTOCOLO_UC_B, 'NULO'), ', ') AS COMBINACOES_TIPO_PROTOCOLO,
    STRING_AGG(DISTINCT COALESCE(PROTOCOLO_A, PROTOCOLO_B), ', ') FILTER (WHERE COALESCE(PROTOCOLO_A, PROTOCOLO_B) IS NOT NULL) AS PROTOCOLOS_ENVOLVIDOS
FROM exploratoria_sobreposicao_tipo_protocolo
GROUP BY UC, CONJUNTO, REGIONAL
ORDER BY HORAS_SOBREPOSTAS DESC, PARES_SOBREPOSTOS DESC
LIMIT 200
""").fetchdf()
sobreposicao_por_uc

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,UC,CONJUNTO,REGIONAL,PARES_SOBREPOSTOS,HORAS_SOBREPOSTAS,COMBINACOES_TIPO_PROTOCOLO,PROTOCOLOS_ENVOLVIDOS
0,7356005,15970,LES,2,22.851389,0 x 1,2026DC00722
1,96824760,14672,NRO,1,14.916667,0 x 1,2026DC00679
2,32412169,16774,LES,1,9.277500,1 x 0,2026DC00679
3,76386953,14699,NRT,1,8.183333,0 x 1,2026DC00679
4,54748194,14647,NRO,2,6.765556,0 x 1,2026DC00741
...,...,...,...,...,...,...,...
195,53476808,16768,OES,1,2.909722,0 x 1,2026DC00737
196,37186604,16768,OES,1,2.909722,1 x 0,2026DC00737
197,30332664,16768,OES,1,2.909722,1 x 0,2026DC00737
198,37088408,16768,OES,1,2.909722,1 x 0,2026DC00737


## 4. Mesmo dia e conjunto com UCs com e sem protocolo

Procura `CONJUNTO + DATA` onde existam simultaneamente:

- UCs com protocolo recuperado;
- UCs sem protocolo recuperado.

Também marca UCs que, no mesmo dia, possuem linhas com e sem protocolo.

In [9]:
con.execute("""
CREATE OR REPLACE TEMP VIEW exploratoria_conjunto_dia_protocolo AS
WITH uc_dia AS (
    SELECT
        CONJUNTO,
        REGIONAL,
        DATA_INTRP,
        UC,
        MAX(CASE WHEN TEM_PROTOCOLO = 'S' THEN 1 ELSE 0 END) AS UC_TEM_PROTOCOLO,
        MAX(CASE WHEN TEM_PROTOCOLO = 'N' THEN 1 ELSE 0 END) AS UC_TEM_LINHA_SEM_PROTOCOLO,
        COUNT(*) AS LINHAS_UC_DIA,
        STRING_AGG(DISTINCT PROTOCOLO_RECUPERADO, ', ' ORDER BY PROTOCOLO_RECUPERADO) FILTER (WHERE PROTOCOLO_RECUPERADO IS NOT NULL) AS PROTOCOLOS_UC,
        STRING_AGG(DISTINCT COALESCE(TIPO_PROTOCOLO_UC, 'NULO'), ', ' ORDER BY COALESCE(TIPO_PROTOCOLO_UC, 'NULO')) AS TIPOS_PROTOCOLO_UC
    FROM base_eventos_exploratoria
    GROUP BY CONJUNTO, REGIONAL, DATA_INTRP, UC
), conjunto_dia AS (
    SELECT
        CONJUNTO,
        REGIONAL,
        DATA_INTRP,
        COUNT(DISTINCT UC) AS UCS_NO_CONJUNTO_DIA,
        SUM(UC_TEM_PROTOCOLO) AS UCS_COM_PROTOCOLO,
        SUM(UC_TEM_LINHA_SEM_PROTOCOLO) AS UCS_COM_LINHA_SEM_PROTOCOLO,
        SUM(CASE WHEN UC_TEM_PROTOCOLO = 1 AND UC_TEM_LINHA_SEM_PROTOCOLO = 1 THEN 1 ELSE 0 END) AS UCS_COM_E_SEM_PROTOCOLO,
        SUM(LINHAS_UC_DIA) AS LINHAS_EVENTO,
        STRING_AGG(DISTINCT PROTOCOLOS_UC, ', ') FILTER (WHERE PROTOCOLOS_UC IS NOT NULL) AS PROTOCOLOS_DO_CONJUNTO_DIA
    FROM uc_dia
    GROUP BY CONJUNTO, REGIONAL, DATA_INTRP
)
SELECT *
FROM conjunto_dia
WHERE UCS_COM_PROTOCOLO > 0
  AND UCS_COM_LINHA_SEM_PROTOCOLO > 0
""")

con.execute("""
SELECT
    COUNT(*) AS CONJUNTOS_DIA_COM_MISTURA,
    SUM(UCS_NO_CONJUNTO_DIA) AS UCS_ENVOLVIDAS,
    SUM(UCS_COM_PROTOCOLO) AS UCS_COM_PROTOCOLO,
    SUM(UCS_COM_LINHA_SEM_PROTOCOLO) AS UCS_COM_LINHA_SEM_PROTOCOLO,
    SUM(UCS_COM_E_SEM_PROTOCOLO) AS UCS_COM_E_SEM_PROTOCOLO
FROM exploratoria_conjunto_dia_protocolo
""").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,CONJUNTOS_DIA_COM_MISTURA,UCS_ENVOLVIDAS,UCS_COM_PROTOCOLO,UCS_COM_LINHA_SEM_PROTOCOLO,UCS_COM_E_SEM_PROTOCOLO
0,229,1201857.0,394322.0,819423.0,11888.0


In [10]:
conjunto_dia_amostra = con.execute("""
SELECT *
FROM exploratoria_conjunto_dia_protocolo
ORDER BY UCS_COM_LINHA_SEM_PROTOCOLO DESC, UCS_COM_PROTOCOLO DESC, DATA_INTRP, CONJUNTO
LIMIT 200
""").fetchdf()
conjunto_dia_amostra

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,CONJUNTO,REGIONAL,DATA_INTRP,UCS_NO_CONJUNTO_DIA,UCS_COM_PROTOCOLO,UCS_COM_LINHA_SEM_PROTOCOLO,UCS_COM_E_SEM_PROTOCOLO,LINHAS_EVENTO,PROTOCOLOS_DO_CONJUNTO_DIA
0,14672,NRO,2026-06-22,22819,349.0,22470.0,0.0,24095.0,2026DC00722
1,16755,NRT,2026-06-19,18724,2.0,18722.0,0.0,23256.0,2026DC00716
2,15576,OES,2026-06-24,17511,4.0,17507.0,0.0,17644.0,2026DC00732
3,15939,OES,2026-06-02,15577,390.0,15577.0,390.0,29030.0,202600029
4,14672,NRO,2026-06-11,15186,12.0,15174.0,0.0,23520.0,2026DC00681
...,...,...,...,...,...,...,...,...,...
195,14603,LES,2026-06-20,175,28.0,147.0,0.0,175.0,2026DC00720
196,15951,LES,2026-06-24,146,1.0,145.0,0.0,147.0,2026DC00732
197,14634,NRO,2026-06-22,6987,6949.0,141.0,103.0,7452.0,2026DC00722
198,15564,LES,2026-06-12,4983,4847.0,136.0,0.0,4983.0,2026DC00679


## 5. Detalhe de um conjunto/dia

Preencha `CONJUNTO_ANALISE` e `DATA_ANALISE` com valores retornados em `conjunto_dia_amostra`.

In [11]:
CONJUNTO_ANALISE = None  # exemplo: '14586'
DATA_ANALISE = None      # exemplo: '2026-06-29'

if CONJUNTO_ANALISE and DATA_ANALISE:
    detalhe_conjunto_dia = con.execute(
        """
        SELECT
            CONJUNTO,
            REGIONAL,
            DATA_INTRP,
            UC,
            NUM_OCORRENCIA_ADMS,
            NUM_SEQ_INTRP,
            NUM_INTRP_UCI,
            NUM_POSTO_UCI,
            INICIO,
            FIM,
            TIPO_PROTOCOLO_UC,
            TIPO_PROTOCOLO_INTRP,
            PROTOCOLO_UC,
            PROTOCOLO_INTRP,
            PROTOCOLO_RECUPERADO,
            TEM_PROTOCOLO,
            COD_CAUSA_INTRP,
            COD_COMP_INTRP,
            DURACAO_HORA,
            CI_LIQUIDO,
            CHI_LIQUIDO
        FROM base_eventos_exploratoria
        WHERE CONJUNTO = ?
          AND DATA_INTRP = CAST(? AS DATE)
        ORDER BY UC, INICIO, NUM_SEQ_INTRP, NUM_INTRP_UCI
        """,
        [str(CONJUNTO_ANALISE), DATA_ANALISE],
    ).fetchdf()
else:
    detalhe_conjunto_dia = pd.DataFrame()

detalhe_conjunto_dia

""


## 6. Investigação direta por UC/data

Útil para validar casos conhecidos, como UC que aparece no IQS com protocolo em algumas interrupções e sem protocolo em outras no mesmo dia.

In [12]:
UC_ANALISE = None    # exemplo: '43302351'
DATA_UC_ANALISE = None  # exemplo: '2026-06-29'

if UC_ANALISE and DATA_UC_ANALISE:
    detalhe_uc_dia = con.execute(
        """
        SELECT
            UC,
            CONJUNTO,
            REGIONAL,
            DATA_INTRP,
            NUM_OCORRENCIA_ADMS,
            NUM_SEQ_INTRP,
            NUM_INTRP_UCI,
            INICIO,
            FIM,
            TIPO_PROTOCOLO_UC,
            TIPO_PROTOCOLO_INTRP,
            PROTOCOLO_UC,
            PROTOCOLO_INTRP,
            PROTOCOLO_RECUPERADO,
            TEM_PROTOCOLO,
            COD_CAUSA_INTRP,
            COD_COMP_INTRP,
            DURACAO_HORA
        FROM base_eventos_exploratoria
        WHERE UC = ?
          AND DATA_INTRP = CAST(? AS DATE)
        ORDER BY INICIO, NUM_SEQ_INTRP, NUM_INTRP_UCI
        """,
        [str(UC_ANALISE), DATA_UC_ANALISE],
    ).fetchdf()
else:
    detalhe_uc_dia = pd.DataFrame()

detalhe_uc_dia

""


## 7. Exportações CSV

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d%H%M%S")

arquivos = {
    "sobreposicao_tipo_protocolo": MARTS_DIR / f"Exploratoria_Sobreposicao_Tipo_Protocolo_{ANOMES}_{timestamp}.CSV",
    "sobreposicao_por_uc": MARTS_DIR / f"Exploratoria_Sobreposicao_Tipo_Protocolo_UC_{ANOMES}_{timestamp}.CSV",
    "conjunto_dia_protocolo": MARTS_DIR / f"Exploratoria_Conjunto_Dia_Protocolo_{ANOMES}_{timestamp}.CSV",
}

con.execute(f"COPY exploratoria_sobreposicao_tipo_protocolo TO '{arquivos['sobreposicao_tipo_protocolo'].as_posix()}' WITH (HEADER TRUE, DELIMITER '|')")
sobreposicao_por_uc.to_csv(arquivos["sobreposicao_por_uc"], sep="|", index=False, encoding="utf-8")
con.execute(f"COPY exploratoria_conjunto_dia_protocolo TO '{arquivos['conjunto_dia_protocolo'].as_posix()}' WITH (HEADER TRUE, DELIMITER '|')")

arquivos

## 8. Leituras sugeridas

- Se houver muitos casos de sobreposição com tipos diferentes, separar por combinação `0 x 1`, `0 x 5`, etc.
- Se a regra evoluir para recuperação de protocolo, decidir se a chave será mesma UC/dia ou conjunto/dia.
- Validar se o protocolo recuperado deve priorizar `NUM_PROTOC_JUSTIF_RESP_UCI` ou `NUM_PROTOC_JUSTIF_RESP_INTRP`.
- Manter esta etapa como auditoria até a regra de negócio estar fechada.

## Leitura de dados Serviço ADMS pasta P:\Common\IQS\ADMS\Backup


In [23]:
from pathlib import Path
import pandas as pd

BACKUP_DIR = Path(r"P:\Common\IQS\ADMS\Backup")
ANOMES_BACKUP = "202606"

DATA_INICIO_BACKUP = pd.Timestamp(f"{ANOMES_BACKUP[:4]}-{ANOMES_BACKUP[4:6]}-01")
DATA_FIM_BACKUP = DATA_INICIO_BACKUP + pd.offsets.MonthBegin(1)

assert BACKUP_DIR.exists(), f"Pasta não encontrada: {BACKUP_DIR}"

# Ajuste os padrões se o nome real do arquivo tiver prefixo/sufixo específico
padroes = [
    "IQS_SERVICOS_*.CSV",
    "IQS_SERVICOS_*.csv",
]

arquivos = []
for padrao in padroes:
    arquivos.extend(BACKUP_DIR.glob(padrao))

arquivos = sorted(set(arquivos))

print(f"Arquivos encontrados: {len(arquivos)}")
for arquivo in arquivos[:20]:
    print(arquivo.name)

if not arquivos:
    raise RuntimeError("Nenhum arquivo de junho encontrado. Ajuste os padrões de busca.")

Arquivos encontrados: 21
IQS_SERVICOS_20260626054716.CSV
IQS_SERVICOS_20260627053657.CSV
IQS_SERVICOS_20260627184611.CSV
IQS_SERVICOS_20260628051509.CSV
IQS_SERVICOS_20260628184344.CSV
IQS_SERVICOS_20260629051152.CSV
IQS_SERVICOS_20260629185901.CSV
IQS_SERVICOS_20260630052111.CSV
IQS_SERVICOS_20260630193004.CSV
IQS_SERVICOS_20260701045350.CSV
IQS_SERVICOS_20260701185625.CSV
IQS_SERVICOS_20260702050306.CSV
IQS_SERVICOS_20260702190128.CSV
IQS_SERVICOS_20260703051809.CSV
IQS_SERVICOS_20260703185448.CSV
IQS_SERVICOS_20260704052324.CSV
IQS_SERVICOS_20260704185137.CSV
IQS_SERVICOS_20260705052550.CSV
IQS_SERVICOS_20260705190625.CSV
IQS_SERVICOS_20260706045501.CSV


In [24]:
colunas_data = [
    "DTHR_ALT_ACES",
    "DTHR_SOLIC_SRV",
    "DTHR_GERA_SRV",
    "DTHR_PREV_EXEC_SRV",
    "DTHR_PROG_EXEC_SRV",
    "DTHR_DESPACH_SRV",
    "DTHR_ULT_ALT_SRV",
    "DTHR_FECH_SRV",
    "DTHR_SAIDA_SRV",
    "DTHR_INIC_SRV",
    "DTHR_TERM_SRV",
    "DTHR_RETOR_SRV",
    "DTHR_AGUARD_EQUIPE_SRV",
    "DTHR_REENVIO_SRV",
]

dfs = []

for arquivo in arquivos:
    print(f"Lendo: {arquivo.name}")

    df = pd.read_csv(
        arquivo,
        sep="|",
        dtype=str,
        encoding="utf-8",
        engine="python",
        keep_default_na=False,
    )

    df.columns = [col.strip().upper() for col in df.columns]

    for coluna in df.columns:
        df[coluna] = df[coluna].astype(str).str.strip()

    for coluna in colunas_data:
        if coluna in df.columns:
            df[coluna] = pd.to_datetime(df[coluna], errors="coerce")

    df["ARQUIVO_ORIGEM"] = arquivo.name

    # Filtro principal de junho pelo início do serviço
    if "DTHR_INIC_SRV" in df.columns:
        df = df[
            (df["DTHR_INIC_SRV"] >= DATA_INICIO_BACKUP)
            & (df["DTHR_INIC_SRV"] < DATA_FIM_BACKUP)
        ]

    dfs.append(df)

backup_adms_servicos_junho = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()

print("Linhas carregadas:", len(backup_adms_servicos_junho))
backup_adms_servicos_junho.head()

Lendo: IQS_SERVICOS_20260626054716.CSV
Lendo: IQS_SERVICOS_20260627053657.CSV
Lendo: IQS_SERVICOS_20260627184611.CSV
Lendo: IQS_SERVICOS_20260628051509.CSV
Lendo: IQS_SERVICOS_20260628184344.CSV
Lendo: IQS_SERVICOS_20260629051152.CSV
Lendo: IQS_SERVICOS_20260629185901.CSV
Lendo: IQS_SERVICOS_20260630052111.CSV
Lendo: IQS_SERVICOS_20260630193004.CSV
Lendo: IQS_SERVICOS_20260701045350.CSV
Lendo: IQS_SERVICOS_20260701185625.CSV
Lendo: IQS_SERVICOS_20260702050306.CSV
Lendo: IQS_SERVICOS_20260702190128.CSV
Lendo: IQS_SERVICOS_20260703051809.CSV
Lendo: IQS_SERVICOS_20260703185448.CSV
Lendo: IQS_SERVICOS_20260704052324.CSV
Lendo: IQS_SERVICOS_20260704185137.CSV
Lendo: IQS_SERVICOS_20260705052550.CSV
Lendo: IQS_SERVICOS_20260705190625.CSV
Lendo: IQS_SERVICOS_20260706045501.CSV
Lendo: IQS_SERVICOS_20260707050819.CSV
Linhas carregadas: 26263


,DTHR_ALT_ACES,PID,INDIC_EST_SERV_SRV,NUM_ORG_EXEC_SRV,DTHR_SOLIC_SRV,DTHR_GERA_SRV,DTHR_PREV_EXEC_SRV,DTHR_PROG_EXEC_SRV,DTHR_DESPACH_SRV,DTHR_ULT_ALT_SRV,...,COD_CAUSA_SRVE,COD_COMP_SRVE,COD_COND_CLIMA_SRVE,COD_LOCAL_CIS_SRV,PID_INTRP_SRVE,DTHR_REENVIO_SRV,COD_LOCAL_ATEND_SRV,COD_CONJTO_ELET_ANEEL,ESTADO_SERVICO_ACOMP,ARQUIVO_ORIGEM
0,2026-06-26 05:47:24,126316453,14,114615,2026-06-10 00:48:00,2026-06-10 00:49:52,2026-06-10 04:55:00,2026-06-10 04:51:48,2026-06-10 02:39:00,2026-06-25 14:41:03,...,22,52,1,3046,,NaT,3046,15938,14,IQS_SERVICOS_20260626054716.CSV
1,2026-06-26 05:47:24,126373748,14,114100,2026-06-24 15:44:00,2026-06-24 15:58:52,2026-06-24 13:00:00,2026-06-24 18:02:05,2026-06-24 16:01:00,2026-06-24 19:41:59,...,39,89,1,6478,261105501111055070,NaT,2458,14682,14,IQS_SERVICOS_20260626054716.CSV
2,2026-06-26 05:47:24,126373745,14,114061,2026-06-24 15:44:00,2026-06-24 15:58:33,2026-06-24 13:00:00,2026-06-24 17:46:32,2026-06-24 16:03:00,2026-06-24 20:34:30,...,39,89,1,6478,261105501111055070,NaT,2212,14682,14,IQS_SERVICOS_20260626054716.CSV
3,2026-06-26 05:47:24,126366939,14,114370,2026-06-23 00:39:40,2026-06-23 00:39:40,2026-06-23 18:00:00,2026-06-23 16:39:29,2026-06-23 08:08:00,2026-06-25 13:54:17,...,39,89,1,4750,,NaT,4066,14642,14,IQS_SERVICOS_20260626054716.CSV
4,2026-06-26 05:47:24,126366936,14,115753,2026-06-23 00:39:12,2026-06-23 00:39:12,2026-06-23 18:00:00,2026-06-23 11:30:11,2026-06-23 08:07:00,2026-06-25 13:53:01,...,39,89,1,4750,,NaT,4750,14642,14,IQS_SERVICOS_20260626054716.CSV


In [25]:
con.register("backup_adms_servicos_junho_df", backup_adms_servicos_junho)

con.execute("""
CREATE OR REPLACE TEMP VIEW backup_adms_servicos_junho AS
SELECT *
FROM backup_adms_servicos_junho_df
""")

con.execute("""
SELECT
    COUNT(*) AS LINHAS,
    COUNT(DISTINCT PID_INTRP_SRVE) AS INTERRUPCOES,
    COUNT(DISTINCT NUM_SEQ_SERV) AS SERVICOS,
    MIN(DTHR_INIC_SRV) AS MENOR_INICIO,
    MAX(DTHR_INIC_SRV) AS MAIOR_INICIO
FROM backup_adms_servicos_junho
""").fetchdf()

,LINHAS,INTERRUPCOES,SERVICOS,MENOR_INICIO,MAIOR_INICIO
0,26263,20026,23744,2026-06-01 07:58:00,2026-06-30 23:58:00


In [26]:
servicos_por_interrupcao = con.execute("""
SELECT
    NULLIF(TRIM(CAST(PID_INTRP_SRVE AS VARCHAR)), '') AS NUM_SEQ_INTRP,
    COUNT(*) AS QTD_SERVICOS_BACKUP,
    STRING_AGG(DISTINCT NULLIF(TRIM(CAST(NUM_SEQ_SERV AS VARCHAR)), ''), ', ') AS NUM_SEQ_SERVICOS,
    STRING_AGG(DISTINCT NULLIF(TRIM(CAST(COD_CAUSA_SRVE AS VARCHAR)), ''), ', ') AS CAUSAS_SERVICO,
    STRING_AGG(DISTINCT NULLIF(TRIM(CAST(COD_COMP_SRVE AS VARCHAR)), ''), ', ') AS COMPONENTES_SERVICO,
    MIN(DTHR_INIC_SRV) AS PRIMEIRO_INICIO_SERVICO,
    MAX(DTHR_TERM_SRV) AS ULTIMO_TERMINO_SERVICO
FROM backup_adms_servicos_junho
WHERE NULLIF(TRIM(CAST(PID_INTRP_SRVE AS VARCHAR)), '') IS NOT NULL
GROUP BY NULLIF(TRIM(CAST(PID_INTRP_SRVE AS VARCHAR)), '')
""").fetchdf()

servicos_por_interrupcao.head()

,NUM_SEQ_INTRP,QTD_SERVICOS_BACKUP,NUM_SEQ_SERVICOS,CAUSAS_SERVICO,COMPONENTES_SERVICO,PRIMEIRO_INICIO_SERVICO,ULTIMO_TERMINO_SERVICO
0,261106761011068867,1,50885403,18,35,2026-06-25 10:31:00,2026-06-25 12:16:00
1,261107140911071410,1,10439501,78,46,2026-06-22 13:56:00,2026-06-22 14:29:00
2,261103860711038628,1,20874757,85,52,2026-06-23 00:14:00,2026-06-23 00:17:00
3,261107149711071499,1,10440575,22,52,2026-06-24 00:26:00,2026-06-24 00:30:00
4,261106540411065405,1,10440596,22,52,2026-06-23 19:40:00,2026-06-23 19:42:00


In [27]:
potencial_chi_tipo_0_enriquecido = con.execute("""
WITH servicos AS (
    SELECT
        NULLIF(TRIM(CAST(PID_INTRP_SRVE AS VARCHAR)), '') AS NUM_SEQ_INTRP,
        COUNT(*) AS QTD_SERVICOS_BACKUP,
        STRING_AGG(DISTINCT NULLIF(TRIM(CAST(NUM_SEQ_SERV AS VARCHAR)), ''), ', ') AS NUM_SEQ_SERVICOS,
        STRING_AGG(DISTINCT NULLIF(TRIM(CAST(COD_CAUSA_SRVE AS VARCHAR)), ''), ', ') AS CAUSAS_SERVICO,
        STRING_AGG(DISTINCT NULLIF(TRIM(CAST(COD_COMP_SRVE AS VARCHAR)), ''), ', ') AS COMPONENTES_SERVICO,
        MIN(DTHR_INIC_SRV) AS PRIMEIRO_INICIO_SERVICO,
        MAX(DTHR_TERM_SRV) AS ULTIMO_TERMINO_SERVICO
    FROM backup_adms_servicos_junho
    WHERE NULLIF(TRIM(CAST(PID_INTRP_SRVE AS VARCHAR)), '') IS NOT NULL
    GROUP BY NULLIF(TRIM(CAST(PID_INTRP_SRVE AS VARCHAR)), '')
)
SELECT
    p.*,
    s.QTD_SERVICOS_BACKUP,
    s.NUM_SEQ_SERVICOS,
    s.CAUSAS_SERVICO,
    s.COMPONENTES_SERVICO,
    s.PRIMEIRO_INICIO_SERVICO,
    s.ULTIMO_TERMINO_SERVICO
FROM potencial_chi_tipo_0 p
LEFT JOIN servicos s
  ON POSITION(s.NUM_SEQ_INTRP IN p.INTERRUPCOES) > 0
""").fetchdf()

potencial_chi_tipo_0_enriquecido.head()

,UC,CONJUNTO,REGIONAL,DATA_INTRP,QTD_REGISTROS_TIPO_0_SOBREPOSTOS,QTD_INTERRUPCOES_TIPO_0,POTENCIAL_CHI_TIPO_0_RECLASSIFICACAO,FATURADA,META_DIC,DIC,...,RESSARCIMENTO_CHI_TIPO_0_ESTIMADO,OCORRENCIAS,INTERRUPCOES,PROTOCOLOS_ENVOLVIDOS,QTD_SERVICOS_BACKUP,NUM_SEQ_SERVICOS,CAUSAS_SERVICO,COMPONENTES_SERVICO,PRIMEIRO_INICIO_SERVICO,ULTIMO_TERMINO_SERVICO
0,77325400,15970,LES,2026-06-28,1,1,24.027222,S,19.0,27.137778,...,2.266765e+06,1-26488056,261113929211139294,None,<NA>,None,None,None,NaT,NaT
1,77325400,15970,LES,2026-06-29,1,1,24.027222,S,19.0,27.137778,...,2.266765e+06,1-26488056,261113929211139294,None,<NA>,None,None,None,NaT,NaT
2,43152538,14658,OES,2026-06-29,1,1,28.461389,S,20.0,33.779722,...,2.208325e+05,1-26489388,261111916711144844,None,<NA>,None,None,None,NaT,NaT
3,113417560,15970,LES,2026-06-29,1,1,24.027222,S,19.0,30.376111,...,2.002231e+05,1-26488056,261113929211139294,None,<NA>,None,None,None,NaT,NaT
4,22090533,15970,LES,2026-06-29,1,1,24.027222,S,19.0,30.376111,...,1.996703e+05,1-26488056,261113929211139294,None,<NA>,None,None,None,NaT,NaT


In [29]:
con.execute("""
CREATE OR REPLACE TEMP VIEW eventos_tipo_0_com_sobreposicao AS
WITH pares_sobrepostos AS (
    SELECT
        a.UC,
        a.CONJUNTO,
        a.REGIONAL,
        a.DATA_INTRP,

        a.ID_EVENTO AS ID_EVENTO_A,
        b.ID_EVENTO AS ID_EVENTO_B,

        a.NUM_OCORRENCIA_ADMS AS OCORRENCIA_A,
        b.NUM_OCORRENCIA_ADMS AS OCORRENCIA_B,

        a.NUM_SEQ_INTRP AS INTERRUPCAO_A,
        b.NUM_SEQ_INTRP AS INTERRUPCAO_B,

        a.NUM_INTRP_UCI AS INTRP_UCI_A,
        b.NUM_INTRP_UCI AS INTRP_UCI_B,

        a.TIPO_PROTOCOLO_UC AS TIPO_PROTOCOLO_UC_A,
        b.TIPO_PROTOCOLO_UC AS TIPO_PROTOCOLO_UC_B,

        a.PROTOCOLO_RECUPERADO AS PROTOCOLO_A,
        b.PROTOCOLO_RECUPERADO AS PROTOCOLO_B,

        a.INICIO AS INICIO_A,
        a.FIM AS FIM_A,
        b.INICIO AS INICIO_B,
        b.FIM AS FIM_B,

        a.CHI_LIQUIDO AS CHI_LIQUIDO_A,
        b.CHI_LIQUIDO AS CHI_LIQUIDO_B

    FROM base_eventos_exploratoria a
    JOIN base_eventos_exploratoria b
      ON a.UC = b.UC
     AND a.ID_EVENTO < b.ID_EVENTO
     AND a.INICIO < b.FIM
     AND a.FIM > b.INICIO
     AND COALESCE(a.TIPO_PROTOCOLO_UC, 'NULO') <> COALESCE(b.TIPO_PROTOCOLO_UC, 'NULO')
)
SELECT DISTINCT
    UC,
    CONJUNTO,
    REGIONAL,
    DATA_INTRP,
    ID_EVENTO_A AS ID_EVENTO,
    OCORRENCIA_A AS NUM_OCORRENCIA_ADMS,
    INTERRUPCAO_A AS NUM_SEQ_INTRP,
    INTRP_UCI_A AS NUM_INTRP_UCI,
    TIPO_PROTOCOLO_UC_A AS TIPO_PROTOCOLO_UC,
    PROTOCOLO_A AS PROTOCOLO_RECUPERADO,
    INICIO_A AS INICIO,
    FIM_A AS FIM,
    CHI_LIQUIDO_A AS CHI_LIQUIDO,
    CHI_LIQUIDO_A AS POTENCIAL_CHI_TIPO_0
FROM pares_sobrepostos
WHERE TIPO_PROTOCOLO_UC_A = '0'

UNION

SELECT DISTINCT
    UC,
    CONJUNTO,
    REGIONAL,
    DATA_INTRP,
    ID_EVENTO_B AS ID_EVENTO,
    OCORRENCIA_B AS NUM_OCORRENCIA_ADMS,
    INTERRUPCAO_B AS NUM_SEQ_INTRP,
    INTRP_UCI_B AS NUM_INTRP_UCI,
    TIPO_PROTOCOLO_UC_B AS TIPO_PROTOCOLO_UC,
    PROTOCOLO_B AS PROTOCOLO_RECUPERADO,
    INICIO_B AS INICIO,
    FIM_B AS FIM,
    CHI_LIQUIDO_B AS CHI_LIQUIDO,
    CHI_LIQUIDO_B AS POTENCIAL_CHI_TIPO_0
FROM pares_sobrepostos
WHERE TIPO_PROTOCOLO_UC_B = '0'
""")

In [32]:
servicos_improcedentes_duplicidade = con.execute("""
WITH servicos_improcedentes AS (
    SELECT
        NULLIF(TRIM(CAST(PID_INTRP_SRVE AS VARCHAR)), '') AS NUM_SEQ_INTRP,
        NULLIF(TRIM(CAST(NUM_SEQ_SERV AS VARCHAR)), '') AS NUM_SEQ_SERV,
        NULLIF(TRIM(CAST(PID AS VARCHAR)), '') AS PID_SERVICO,
        NULLIF(TRIM(CAST(NUM_ORG_EXEC_SRV AS VARCHAR)), '') AS NUM_ORG_EXEC_SRV,
        DTHR_SOLIC_SRV,
        DTHR_GERA_SRV,
        DTHR_INIC_SRV,
        DTHR_TERM_SRV,
        DTHR_FECH_SRV,
        NULLIF(TRIM(CAST(COD_CAUSA_SRVE AS VARCHAR)), '') AS COD_CAUSA_SRVE,
        NULLIF(TRIM(CAST(COD_COMP_SRVE AS VARCHAR)), '') AS COD_COMP_SRVE,
        NULLIF(TRIM(CAST(ESTADO_SERVICO_ACOMP AS VARCHAR)), '') AS ESTADO_SERVICO_ACOMP,
        NULLIF(TRIM(CAST(COD_CONJTO_ELET_ANEEL AS VARCHAR)), '') AS CONJUNTO_SERVICO,
        ARQUIVO_ORIGEM
    FROM backup_adms_servicos_junho
    WHERE LPAD(NULLIF(TRIM(CAST(COD_CAUSA_SRVE AS VARCHAR)), ''), 2, '0') IN ('22', '85')
),
interrupcoes_potencial_tipo_0 AS (
    SELECT DISTINCT
        UC,
        CONJUNTO,
        REGIONAL,
        DATA_INTRP,
        NUM_SEQ_INTRP,
        NUM_OCORRENCIA_ADMS,
        PROTOCOLO_RECUPERADO,
        TIPO_PROTOCOLO_UC,
        INICIO,
        FIM,
        POTENCIAL_CHI_TIPO_0
    FROM eventos_tipo_0_com_sobreposicao
),
servicos_cruzados AS (
    SELECT
        s.*,
        p.UC,
        p.CONJUNTO,
        p.REGIONAL,
        p.DATA_INTRP,
        p.NUM_OCORRENCIA_ADMS,
        p.PROTOCOLO_RECUPERADO,
        p.TIPO_PROTOCOLO_UC,
        p.INICIO AS INICIO_INTERRUPCAO,
        p.FIM AS FIM_INTERRUPCAO,
        p.POTENCIAL_CHI_TIPO_0,
        CASE
            WHEN p.NUM_SEQ_INTRP IS NOT NULL THEN 'S'
            ELSE 'N'
        END AS TEM_DUPLICIDADE_INTERRUPCAO_INDEVIDA
    FROM servicos_improcedentes s
    LEFT JOIN interrupcoes_potencial_tipo_0 p
      ON TRIM(CAST(p.NUM_SEQ_INTRP AS VARCHAR)) = TRIM(CAST(s.NUM_SEQ_INTRP AS VARCHAR))
)
SELECT *
FROM servicos_cruzados
ORDER BY
    TEM_DUPLICIDADE_INTERRUPCAO_INDEVIDA DESC,
    DATA_INTRP,
    NUM_SEQ_INTRP,
    NUM_SEQ_SERV
""").fetchdf()

servicos_improcedentes_duplicidade

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,NUM_SEQ_INTRP,NUM_SEQ_SERV,PID_SERVICO,NUM_ORG_EXEC_SRV,DTHR_SOLIC_SRV,DTHR_GERA_SRV,DTHR_INIC_SRV,DTHR_TERM_SRV,DTHR_FECH_SRV,COD_CAUSA_SRVE,...,CONJUNTO,REGIONAL,DATA_INTRP,NUM_OCORRENCIA_ADMS,PROTOCOLO_RECUPERADO,TIPO_PROTOCOLO_UC,INICIO_INTERRUPCAO,FIM_INTERRUPCAO,POTENCIAL_CHI_TIPO_0,TEM_DUPLICIDADE_INTERRUPCAO_INDEVIDA
0,261111290611112937,50890818,126382743,114061,2026-06-26 21:31:00,2026-06-26 21:32:37,2026-06-28 18:43:00,2026-06-28 18:44:00,2026-06-28 18:44:00,22,...,16760,OES,2026-06-26,1-26474785,None,0,2026-06-26 21:31:00,2026-06-28 18:44:00,45.216667,S
1,261110499411105024,50893485,126388145,114470,2026-06-27 21:38:00,2026-06-27 21:39:07,2026-06-28 10:21:00,2026-06-28 10:23:00,2026-06-28 10:23:00,22,...,15919,OES,2026-06-27,1-26481718,None,0,2026-06-27 21:38:00,2026-06-28 10:23:00,12.750000,S
2,261113174111131752,10442783,126393071,114717,2026-06-28 21:14:00,2026-06-28 21:15:07,2026-06-29 17:31:00,2026-06-29 17:35:00,2026-06-29 17:35:00,22,...,14638,CSL,2026-06-28,1-26487746,None,0,2026-06-28 21:14:00,2026-06-29 17:35:00,20.350000,S
3,261113627011136322,10442825,126393247,114717,2026-06-28 22:09:00,2026-06-28 22:10:37,2026-06-29 21:29:00,2026-06-29 21:33:00,2026-06-29 21:33:00,85,...,15574,CSL,2026-06-28,1-26487948,None,0,2026-06-28 22:09:00,2026-06-29 21:33:00,23.400000,S
4,261113842111138422,40412917,126393353,114892,2026-06-28 23:02:00,2026-06-28 23:03:37,2026-06-29 19:42:00,2026-06-29 19:43:00,2026-06-29 19:43:00,22,...,14632,NRT,2026-06-28,1-26488078,None,0,2026-06-28 23:02:00,2026-06-29 19:43:00,20.683333,S
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5336,None,50898948,126406432,114296,2026-06-30 01:54:00,2026-06-30 17:19:50,2026-06-30 17:57:00,2026-06-30 18:08:00,2026-06-30 18:08:00,22,...,None,None,NaT,None,None,None,NaT,NaT,NaN,N
5337,None,50899091,126406801,114332,2026-06-29 22:37:00,2026-06-30 18:15:38,2026-06-30 18:23:00,2026-06-30 18:29:00,2026-06-30 18:29:00,22,...,None,None,NaT,None,None,None,NaT,NaT,NaN,N
5338,None,50899303,126407340,114453,2026-06-30 15:43:00,2026-06-30 19:24:07,2026-06-30 22:53:00,2026-06-30 22:59:00,2026-06-30 22:59:00,22,...,None,None,NaT,None,None,None,NaT,NaT,NaN,N
5339,None,50899341,126407473,114332,2026-06-30 19:30:00,2026-06-30 19:36:43,2026-06-30 22:07:00,2026-06-30 22:08:00,2026-06-30 22:08:00,22,...,None,None,NaT,None,None,None,NaT,NaT,NaN,N


In [33]:
con.execute("""
SELECT
    COD_CAUSA_SRVE,
    COD_COMP_SRVE,
    ESTADO_SERVICO_ACOMP,
    COUNT(*) AS QTD
FROM backup_adms_servicos_junho
GROUP BY
    COD_CAUSA_SRVE,
    COD_COMP_SRVE,
    ESTADO_SERVICO_ACOMP
ORDER BY QTD DESC
LIMIT 100
""").fetchdf()

,COD_CAUSA_SRVE,COD_COMP_SRVE,ESTADO_SERVICO_ACOMP,QTD
0,22,52,14,4194
1,39,89,14,2393
2,77,52,14,1894
3,71,52,14,1860
4,04,35,14,1349
...,...,...,...,...
95,54,34,14,24
96,13,25,14,23
97,08,50,14,22
98,13,74,14,22


In [34]:
resumo_servicos_22_85 = con.execute("""
WITH servicos_filtrados AS (
    SELECT
        NULLIF(TRIM(CAST(PID_INTRP_SRVE AS VARCHAR)), '') AS NUM_SEQ_INTRP,
        NULLIF(TRIM(CAST(NUM_SEQ_SERV AS VARCHAR)), '') AS NUM_SEQ_SERV,
        LPAD(NULLIF(TRIM(CAST(COD_CAUSA_SRVE AS VARCHAR)), ''), 2, '0') AS COD_CAUSA_SRVE
    FROM backup_adms_servicos_junho
    WHERE LPAD(NULLIF(TRIM(CAST(COD_CAUSA_SRVE AS VARCHAR)), ''), 2, '0') IN ('22', '85')
),
interrupcoes_potencial_tipo_0 AS (
    SELECT DISTINCT NUM_SEQ_INTRP
    FROM eventos_tipo_0_com_sobreposicao
),
cruzado AS (
    SELECT
        s.*,
        CASE
            WHEN p.NUM_SEQ_INTRP IS NOT NULL THEN 'S'
            ELSE 'N'
        END AS TEM_DUPLICIDADE_INTERRUPCAO_INDEVIDA
    FROM servicos_filtrados s
    LEFT JOIN interrupcoes_potencial_tipo_0 p
      ON TRIM(CAST(p.NUM_SEQ_INTRP AS VARCHAR)) = TRIM(CAST(s.NUM_SEQ_INTRP AS VARCHAR))
)
SELECT
    COD_CAUSA_SRVE,
    CASE
        WHEN COD_CAUSA_SRVE = '22' THEN 'ATENDIDO POR OUTRA OCORRENCIA'
        WHEN COD_CAUSA_SRVE = '85' THEN 'IMPROCEDENTE'
    END AS DESC_CAUSA_SRVE,
    COUNT(*) AS SERVICOS,
    COUNT(DISTINCT NUM_SEQ_SERV) AS SERVICOS_DISTINTOS,
    COUNT(DISTINCT NUM_SEQ_INTRP) AS INTERRUPCOES,
    SUM(CASE WHEN TEM_DUPLICIDADE_INTERRUPCAO_INDEVIDA = 'S' THEN 1 ELSE 0 END) AS SERVICOS_COM_DUPLICIDADE,
    COUNT(DISTINCT CASE WHEN TEM_DUPLICIDADE_INTERRUPCAO_INDEVIDA = 'S' THEN NUM_SEQ_SERV END) AS SERVICOS_DISTINTOS_COM_DUPLICIDADE,
    COUNT(DISTINCT CASE WHEN TEM_DUPLICIDADE_INTERRUPCAO_INDEVIDA = 'S' THEN NUM_SEQ_INTRP END) AS INTERRUPCOES_COM_DUPLICIDADE
FROM cruzado
GROUP BY COD_CAUSA_SRVE
ORDER BY COD_CAUSA_SRVE
""").fetchdf()

resumo_servicos_22_85

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,COD_CAUSA_SRVE,DESC_CAUSA_SRVE,SERVICOS,SERVICOS_DISTINTOS,INTERRUPCOES,SERVICOS_COM_DUPLICIDADE,SERVICOS_DISTINTOS_COM_DUPLICIDADE,INTERRUPCOES_COM_DUPLICIDADE
0,22,ATENDIDO POR OUTRA OCORRENCIA,4194,3779,3210,10.0,9,9
1,85,IMPROCEDENTE,1143,1059,961,4.0,4,4


## Duplicidade de Serviço  com ocorrencia

In [38]:
pares_servicos_22_85_todas_interrupcoes_uc = con.execute("""
WITH servicos_22_85 AS (
    SELECT
        NULLIF(TRIM(CAST(PID_INTRP_SRVE AS VARCHAR)), '') AS NUM_SEQ_INTRP,
        NULLIF(TRIM(CAST(NUM_SEQ_SERV AS VARCHAR)), '') AS NUM_SEQ_SERV,
        NULLIF(TRIM(CAST(PID AS VARCHAR)), '') AS PID_SERVICO,
        LPAD(NULLIF(TRIM(CAST(COD_CAUSA_SRVE AS VARCHAR)), ''), 2, '0') AS COD_CAUSA_SRVE,
        CASE
            WHEN LPAD(NULLIF(TRIM(CAST(COD_CAUSA_SRVE AS VARCHAR)), ''), 2, '0') = '22'
            THEN 'ATENDIDO POR OUTRA OCORRENCIA'
            WHEN LPAD(NULLIF(TRIM(CAST(COD_CAUSA_SRVE AS VARCHAR)), ''), 2, '0') = '85'
            THEN 'IMPROCEDENTE'
            ELSE 'OUTRO'
        END AS DESC_CAUSA_SRVE,
        NULLIF(TRIM(CAST(COD_COMP_SRVE AS VARCHAR)), '') AS COD_COMP_SRVE,
        DTHR_SOLIC_SRV,
        DTHR_GERA_SRV,
        DTHR_INIC_SRV,
        DTHR_TERM_SRV,
        DTHR_FECH_SRV,
        NULLIF(TRIM(CAST(COD_CONJTO_ELET_ANEEL AS VARCHAR)), '') AS CONJUNTO_SERVICO,
        ARQUIVO_ORIGEM
    FROM backup_adms_servicos_junho
    WHERE LPAD(NULLIF(TRIM(CAST(COD_CAUSA_SRVE AS VARCHAR)), ''), 2, '0') IN ('22', '85')
),
servicos_todos AS (
    SELECT
        NULLIF(TRIM(CAST(PID_INTRP_SRVE AS VARCHAR)), '') AS NUM_SEQ_INTRP,
        NULLIF(TRIM(CAST(NUM_SEQ_SERV AS VARCHAR)), '') AS NUM_SEQ_SERV,
        NULLIF(TRIM(CAST(PID AS VARCHAR)), '') AS PID_SERVICO,
        LPAD(NULLIF(TRIM(CAST(COD_CAUSA_SRVE AS VARCHAR)), ''), 2, '0') AS COD_CAUSA_SRVE,
        NULLIF(TRIM(CAST(COD_COMP_SRVE AS VARCHAR)), '') AS COD_COMP_SRVE,
        DTHR_SOLIC_SRV,
        DTHR_GERA_SRV,
        DTHR_INIC_SRV,
        DTHR_TERM_SRV,
        DTHR_FECH_SRV,
        NULLIF(TRIM(CAST(COD_CONJTO_ELET_ANEEL AS VARCHAR)), '') AS CONJUNTO_SERVICO
    FROM backup_adms_servicos_junho
    WHERE NULLIF(TRIM(CAST(PID_INTRP_SRVE AS VARCHAR)), '') IS NOT NULL
),
eventos_uc AS (
    SELECT DISTINCT
        UC,
        CONJUNTO,
        REGIONAL,
        DATA_INTRP,
        NUM_OCORRENCIA_ADMS,
        NUM_SEQ_INTRP,
        NUM_INTRP_UCI,
        NUM_POSTO_UCI,
        TIPO_PROTOCOLO_UC,
        TIPO_PROTOCOLO_INTRP,
        PROTOCOLO_UC,
        PROTOCOLO_INTRP,
        PROTOCOLO_RECUPERADO,
        INICIO,
        FIM,
        COD_CAUSA_INTRP,
        COD_COMP_INTRP,
        CHI_LIQUIDO
    FROM base_eventos_exploratoria
),
servicos_22_85_contexto AS (
    SELECT
        e.UC,
        e.CONJUNTO,
        e.REGIONAL,
        e.DATA_INTRP,
        e.NUM_OCORRENCIA_ADMS,
        e.NUM_SEQ_INTRP,
        e.NUM_INTRP_UCI,
        e.NUM_POSTO_UCI,
        e.TIPO_PROTOCOLO_UC,
        e.TIPO_PROTOCOLO_INTRP,
        e.PROTOCOLO_RECUPERADO,
        e.INICIO,
        e.FIM,
        e.COD_CAUSA_INTRP,
        e.COD_COMP_INTRP,
        e.CHI_LIQUIDO,

        s.NUM_SEQ_SERV,
        s.PID_SERVICO,
        s.COD_CAUSA_SRVE,
        s.DESC_CAUSA_SRVE,
        s.COD_COMP_SRVE,
        s.DTHR_SOLIC_SRV,
        s.DTHR_GERA_SRV,
        s.DTHR_INIC_SRV,
        s.DTHR_TERM_SRV,
        s.DTHR_FECH_SRV
    FROM servicos_22_85 s
    JOIN eventos_uc e
      ON TRIM(CAST(e.NUM_SEQ_INTRP AS VARCHAR)) = TRIM(CAST(s.NUM_SEQ_INTRP AS VARCHAR))
),
pares_candidatos AS (
    SELECT
        base.UC,
        base.CONJUNTO,
        base.REGIONAL,
        base.DATA_INTRP,

        base.NUM_SEQ_SERV AS SERVICO_22_85,
        base.PID_SERVICO AS PID_SERVICO_22_85,
        base.COD_CAUSA_SRVE AS COD_CAUSA_SERVICO_22_85,
        base.DESC_CAUSA_SRVE AS DESC_CAUSA_SERVICO_22_85,

        base.NUM_OCORRENCIA_ADMS AS OCORRENCIA_22_85,
        base.NUM_SEQ_INTRP AS INTERRUPCAO_22_85,
        base.NUM_INTRP_UCI AS INTRP_UCI_22_85,
        base.TIPO_PROTOCOLO_UC AS TIPO_PROTOCOLO_UC_22_85,
        base.TIPO_PROTOCOLO_INTRP AS TIPO_PROTOCOLO_INTRP_22_85,
        base.PROTOCOLO_RECUPERADO AS PROTOCOLO_22_85,
        base.INICIO AS INICIO_INTERRUPCAO_22_85,
        base.FIM AS FIM_INTERRUPCAO_22_85,
        base.COD_CAUSA_INTRP AS COD_CAUSA_INTERRUPCAO_22_85,
        base.COD_COMP_INTRP AS COD_COMP_INTERRUPCAO_22_85,
        base.CHI_LIQUIDO AS CHI_LIQUIDO_22_85,

        par.NUM_OCORRENCIA_ADMS AS OCORRENCIA_PAR,
        par.NUM_SEQ_INTRP AS INTERRUPCAO_PAR,
        par.NUM_INTRP_UCI AS INTRP_UCI_PAR,
        par.TIPO_PROTOCOLO_UC AS TIPO_PROTOCOLO_UC_PAR,
        par.TIPO_PROTOCOLO_INTRP AS TIPO_PROTOCOLO_INTRP_PAR,
        par.PROTOCOLO_RECUPERADO AS PROTOCOLO_PAR,
        par.INICIO AS INICIO_INTERRUPCAO_PAR,
        par.FIM AS FIM_INTERRUPCAO_PAR,
        par.COD_CAUSA_INTRP AS COD_CAUSA_INTERRUPCAO_PAR,
        par.COD_COMP_INTRP AS COD_COMP_INTERRUPCAO_PAR,
        par.CHI_LIQUIDO AS CHI_LIQUIDO_PAR,

        st.NUM_SEQ_SERV AS SERVICO_PAR,
        st.PID_SERVICO AS PID_SERVICO_PAR,
        st.COD_CAUSA_SRVE AS COD_CAUSA_SERVICO_PAR,
        st.COD_COMP_SRVE AS COD_COMP_SERVICO_PAR,
        st.DTHR_SOLIC_SRV AS DTHR_SOLIC_SERVICO_PAR,
        st.DTHR_INIC_SRV AS DTHR_INIC_SERVICO_PAR,
        st.DTHR_FECH_SRV AS DTHR_FECH_SERVICO_PAR,

        CASE
            WHEN base.INICIO < par.FIM
             AND base.FIM > par.INICIO
            THEN 'S'
            ELSE 'N'
        END AS TEM_SOBREPOSICAO_INTERRUPCAO,

        CASE
            WHEN base.INICIO < par.FIM
             AND base.FIM > par.INICIO
            THEN DATE_DIFF(
                'second',
                GREATEST(base.INICIO, par.INICIO),
                LEAST(base.FIM, par.FIM)
            ) / 3600.0
            ELSE 0
        END AS HORAS_SOBREPOSTAS,

        ABS(DATE_DIFF('minute', base.INICIO, par.INICIO)) AS DIFERENCA_INICIO_MINUTOS

    FROM servicos_22_85_contexto base
    JOIN eventos_uc par
      ON par.UC = base.UC
     AND par.DATA_INTRP = base.DATA_INTRP
     AND par.NUM_SEQ_INTRP <> base.NUM_SEQ_INTRP
    LEFT JOIN servicos_todos st
      ON TRIM(CAST(st.NUM_SEQ_INTRP AS VARCHAR)) = TRIM(CAST(par.NUM_SEQ_INTRP AS VARCHAR))
),
ranqueado AS (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY SERVICO_22_85, INTERRUPCAO_22_85
            ORDER BY
                CASE WHEN TEM_SOBREPOSICAO_INTERRUPCAO = 'S' THEN 0 ELSE 1 END,
                HORAS_SOBREPOSTAS DESC,
                CASE WHEN PROTOCOLO_PAR IS NOT NULL THEN 0 ELSE 1 END,
                DIFERENCA_INICIO_MINUTOS ASC,
                INTERRUPCAO_PAR
        ) AS RANK_PAR
    FROM pares_candidatos
)
SELECT *
FROM ranqueado
WHERE RANK_PAR <= 5
ORDER BY
    UC,
    DATA_INTRP,
    SERVICO_22_85,
    RANK_PAR
""").fetchdf()

pares_servicos_22_85_todas_interrupcoes_uc

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,UC,CONJUNTO,REGIONAL,DATA_INTRP,SERVICO_22_85,PID_SERVICO_22_85,COD_CAUSA_SERVICO_22_85,DESC_CAUSA_SERVICO_22_85,OCORRENCIA_22_85,INTERRUPCAO_22_85,...,PID_SERVICO_PAR,COD_CAUSA_SERVICO_PAR,COD_COMP_SERVICO_PAR,DTHR_SOLIC_SERVICO_PAR,DTHR_INIC_SERVICO_PAR,DTHR_FECH_SERVICO_PAR,TEM_SOBREPOSICAO_INTERRUPCAO,HORAS_SOBREPOSTAS,DIFERENCA_INICIO_MINUTOS,RANK_PAR
0,100264212,16760,OES,2026-06-27,50891023,126383552,22,ATENDIDO POR OUTRA OCORRENCIA,1-26480749,261110603911106051,...,None,None,None,NaT,NaT,NaT,N,0.0,117,2
1,100448780,15573,CSL,2026-06-29,10444518,126400657,85,IMPROCEDENTE,1-26494202,261113068111137998,...,None,None,None,NaT,NaT,NaT,N,0.0,9,2
2,101011792,14642,OES,2026-06-27,50892845,126387097,22,ATENDIDO POR OUTRA OCORRENCIA,1-26480187,261118950311189518,...,None,None,None,NaT,NaT,NaT,N,0.0,198,1
3,101351798,14652,LES,2026-06-28,20881064,126390904,85,IMPROCEDENTE,1-26485319,261111273711115949,...,126388762,39,89,2026-06-28 07:25:00,2026-06-28 13:30:00,2026-06-28 15:21:00,N,0.0,502,4
4,101568274,16764,NRO,2026-06-24,30496141,126375690,22,ATENDIDO POR OUTRA OCORRENCIA,1-26465652,261106430511064310,...,126375733,13,40,2026-06-24 21:58:00,2026-06-25 01:15:00,2026-06-25 01:55:00,N,0.0,213,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
345,98986554,14578,LES,2026-06-29,20882284,126395715,22,ATENDIDO POR OUTRA OCORRENCIA,1-26491032,261119435211194353,...,None,None,None,NaT,NaT,NaT,N,0.0,549,3
346,99167557,15947,CSL,2026-06-29,10442979,126393825,85,IMPROCEDENTE,1-26488611,261111717611117185,...,None,None,None,NaT,NaT,NaT,N,0.0,3,5
347,99200112,14657,LES,2026-06-29,20883988,126400295,22,ATENDIDO POR OUTRA OCORRENCIA,1-26495298,261113135111138110,...,None,None,None,NaT,NaT,NaT,N,0.0,266,3
348,99225174,15959,CSL,2026-06-30,10444950,126402687,22,ATENDIDO POR OUTRA OCORRENCIA,1-26499813,261114302011144537,...,None,None,None,NaT,NaT,NaT,N,0.0,265,2


In [44]:
servicos_fechados_em_sequencia_10min = con.execute("""
WITH servicos_base AS (
    SELECT
        NULLIF(TRIM(CAST(PID_INTRP_SRVE AS VARCHAR)), '') AS NUM_SEQ_INTRP,
        NULLIF(TRIM(CAST(NUM_SEQ_SERV AS VARCHAR)), '') AS NUM_SEQ_SERV,
        NULLIF(TRIM(CAST(PID AS VARCHAR)), '') AS PID_SERVICO,
        LPAD(NULLIF(TRIM(CAST(COD_CAUSA_SRVE AS VARCHAR)), ''), 2, '0') AS COD_CAUSA_SRVE,
        NULLIF(TRIM(CAST(COD_COMP_SRVE AS VARCHAR)), '') AS COD_COMP_SRVE,
        NULLIF(TRIM(CAST(COD_CONJTO_ELET_ANEEL AS VARCHAR)), '') AS CONJUNTO_SERVICO,
        NULLIF(TRIM(CAST(ESTADO_SERVICO_ACOMP AS VARCHAR)), '') AS ESTADO_SERVICO_ACOMP,
        NULLIF(TRIM(CAST(NUM_ORG_EXEC_SRV AS VARCHAR)), '') AS NUM_ORG_EXEC_SRV,
        DTHR_SOLIC_SRV,
        DTHR_GERA_SRV,
        DTHR_DESPACH_SRV,
        DTHR_SAIDA_SRV,
        DTHR_INIC_SRV,
        DTHR_TERM_SRV,
        DTHR_RETOR_SRV,
        DTHR_FECH_SRV,
        ARQUIVO_ORIGEM
    FROM backup_adms_servicos_junho
    WHERE NULLIF(TRIM(CAST(PID_INTRP_SRVE AS VARCHAR)), '') IS NOT NULL
),
eventos_uc AS (
    SELECT DISTINCT
        UC,
        CONJUNTO,
        REGIONAL,
        DATA_INTRP,
        NUM_OCORRENCIA_ADMS,
        NUM_SEQ_INTRP,
        NUM_INTRP_UCI,
        INICIO,
        FIM,
        TIPO_PROTOCOLO_UC,
        PROTOCOLO_RECUPERADO,
        COD_CAUSA_INTRP,
        COD_COMP_INTRP
    FROM base_eventos_exploratoria
),
servicos_contexto AS (
    SELECT
        e.UC,
        e.CONJUNTO,
        e.REGIONAL,
        e.DATA_INTRP,
        e.NUM_OCORRENCIA_ADMS,
        e.NUM_SEQ_INTRP,
        e.NUM_INTRP_UCI,
        e.INICIO AS INICIO_INTERRUPCAO,
        e.FIM AS FIM_INTERRUPCAO,
        e.TIPO_PROTOCOLO_UC,
        e.PROTOCOLO_RECUPERADO,
        e.COD_CAUSA_INTRP,
        e.COD_COMP_INTRP,

        s.NUM_SEQ_SERV,
        s.PID_SERVICO,
        s.COD_CAUSA_SRVE,
        s.COD_COMP_SRVE,
        s.ESTADO_SERVICO_ACOMP,
        s.NUM_ORG_EXEC_SRV,
        s.DTHR_SOLIC_SRV,
        s.DTHR_GERA_SRV,
        s.DTHR_DESPACH_SRV,
        s.DTHR_SAIDA_SRV,
        s.DTHR_INIC_SRV,
        s.DTHR_TERM_SRV,
        s.DTHR_RETOR_SRV,
        s.DTHR_FECH_SRV,
        s.ARQUIVO_ORIGEM
    FROM servicos_base s
    JOIN eventos_uc e
      ON TRIM(CAST(e.NUM_SEQ_INTRP AS VARCHAR)) = TRIM(CAST(s.NUM_SEQ_INTRP AS VARCHAR))
),
pares_sequencia AS (
    SELECT
        --a.UC,
        a.CONJUNTO,
        a.REGIONAL,
        a.DATA_INTRP,

        a.NUM_SEQ_SERV AS SERVICO_1,
        b.NUM_SEQ_SERV AS SERVICO_2,

        a.PID_SERVICO AS PID_SERVICO_1,
        b.PID_SERVICO AS PID_SERVICO_2,

        a.NUM_OCORRENCIA_ADMS AS OCORRENCIA_1,
        b.NUM_OCORRENCIA_ADMS AS OCORRENCIA_2,

        a.NUM_SEQ_INTRP AS INTERRUPCAO_1,
        b.NUM_SEQ_INTRP AS INTERRUPCAO_2,

        a.NUM_INTRP_UCI AS INTRP_UCI_1,
        b.NUM_INTRP_UCI AS INTRP_UCI_2,

        a.COD_CAUSA_SRVE AS COD_CAUSA_SERVICO_1,
        b.COD_CAUSA_SRVE AS COD_CAUSA_SERVICO_2,

        a.COD_COMP_SRVE AS COD_COMP_SERVICO_1,
        b.COD_COMP_SRVE AS COD_COMP_SERVICO_2,

        a.ESTADO_SERVICO_ACOMP AS ESTADO_SERVICO_1,
        b.ESTADO_SERVICO_ACOMP AS ESTADO_SERVICO_2,

        a.DTHR_INIC_SRV AS INICIO_SERVICO_1,
        a.DTHR_TERM_SRV AS TERMINO_SERVICO_1,
        a.DTHR_FECH_SRV AS FECHAMENTO_SERVICO_1,

        b.DTHR_INIC_SRV AS INICIO_SERVICO_2,
        b.DTHR_TERM_SRV AS TERMINO_SERVICO_2,
        b.DTHR_FECH_SRV AS FECHAMENTO_SERVICO_2,

        DATE_DIFF('minute', a.DTHR_FECH_SRV, b.DTHR_INIC_SRV) AS MIN_FECH_1_ATE_INIC_2,
        DATE_DIFF('minute', a.DTHR_FECH_SRV, b.DTHR_FECH_SRV) AS MIN_FECH_1_ATE_FECH_2,

        a.TIPO_PROTOCOLO_UC AS TIPO_PROTOCOLO_1,
        b.TIPO_PROTOCOLO_UC AS TIPO_PROTOCOLO_2,

        a.PROTOCOLO_RECUPERADO AS PROTOCOLO_1,
        b.PROTOCOLO_RECUPERADO AS PROTOCOLO_2,

        a.INICIO_INTERRUPCAO AS INICIO_INTERRUPCAO_1,
        a.FIM_INTERRUPCAO AS FIM_INTERRUPCAO_1,
        b.INICIO_INTERRUPCAO AS INICIO_INTERRUPCAO_2,
        b.FIM_INTERRUPCAO AS FIM_INTERRUPCAO_2

    FROM servicos_contexto a
    JOIN servicos_contexto b
      ON a.UC = b.UC
     AND a.DATA_INTRP = b.DATA_INTRP
     AND a.NUM_SEQ_SERV <> b.NUM_SEQ_SERV
     AND a.DTHR_FECH_SRV IS NOT NULL
     AND b.DTHR_INIC_SRV IS NOT NULL
     AND b.DTHR_FECH_SRV IS NOT NULL
     AND b.DTHR_INIC_SRV >= a.DTHR_FECH_SRV
     AND DATE_DIFF('minute', a.DTHR_FECH_SRV, b.DTHR_INIC_SRV) BETWEEN 0 AND 10
)
SELECT *
FROM pares_sequencia
ORDER BY
    --UC,
    DATA_INTRP,
    FECHAMENTO_SERVICO_1,
    INICIO_SERVICO_2
""").fetchdf()

servicos_fechados_em_sequencia_10min

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,CONJUNTO,REGIONAL,DATA_INTRP,SERVICO_1,SERVICO_2,PID_SERVICO_1,PID_SERVICO_2,OCORRENCIA_1,OCORRENCIA_2,INTERRUPCAO_1,...,MIN_FECH_1_ATE_INIC_2,MIN_FECH_1_ATE_FECH_2,TIPO_PROTOCOLO_1,TIPO_PROTOCOLO_2,PROTOCOLO_1,PROTOCOLO_2,INICIO_INTERRUPCAO_1,FIM_INTERRUPCAO_1,INICIO_INTERRUPCAO_2,FIM_INTERRUPCAO_2
0,14632,CSL,2026-06-22,10440223,10440225,126367013,126367015,1-26458356,1-26458356,261104555311046592,...,1,2,1,1,2026DC00722,2026DC00722,2026-06-22 17:16:00,2026-06-23 15:48:00,2026-06-22 17:16:00,2026-06-23 15:48:00
1,14632,CSL,2026-06-22,10440223,10440225,126367013,126367015,1-26458356,1-26458356,261104555311046592,...,1,2,1,1,2026DC00722,2026DC00722,2026-06-22 17:16:00,2026-06-23 15:48:00,2026-06-22 17:16:00,2026-06-23 15:48:00
2,14632,CSL,2026-06-22,10440223,10440225,126367013,126367015,1-26458356,1-26458356,261104555311046592,...,1,2,1,1,2026DC00722,2026DC00722,2026-06-22 17:16:00,2026-06-23 15:48:00,2026-06-22 17:16:00,2026-06-23 15:48:00
3,14632,CSL,2026-06-22,10440223,10440225,126367013,126367015,1-26458356,1-26458356,261104555311046592,...,1,2,1,1,2026DC00722,2026DC00722,2026-06-22 17:16:00,2026-06-23 15:48:00,2026-06-22 17:16:00,2026-06-23 15:48:00
4,14632,CSL,2026-06-22,10440223,10440225,126367013,126367015,1-26458356,1-26458356,261104555311046592,...,1,2,1,1,2026DC00722,2026DC00722,2026-06-22 17:16:00,2026-06-23 15:48:00,2026-06-22 17:16:00,2026-06-23 15:48:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10937,14652,LES,2026-06-30,20885215,20885231,126404528,126404565,1-26501804,1-26501804,261114419411147047,...,5,6,1,1,2026DC00743,2026DC00743,2026-06-30 12:00:08,2026-06-30 15:11:12,2026-06-30 12:00:08,2026-06-30 15:11:12
10938,14652,LES,2026-06-30,20885215,20885231,126404528,126404565,1-26501804,1-26501804,261114419411147047,...,5,6,1,1,2026DC00743,2026DC00743,2026-06-30 12:00:08,2026-06-30 15:11:12,2026-06-30 12:00:08,2026-06-30 15:11:12
10939,14652,LES,2026-06-30,20885215,20885231,126404528,126404565,1-26501804,1-26501804,261114419411147047,...,5,6,1,1,2026DC00743,2026DC00743,2026-06-30 12:00:08,2026-06-30 15:11:12,2026-06-30 12:00:08,2026-06-30 15:11:12
10940,14652,LES,2026-06-30,20885215,20885231,126404528,126404565,1-26501804,1-26501804,261114419411147047,...,5,6,1,1,2026DC00743,2026DC00743,2026-06-30 12:00:08,2026-06-30 15:11:12,2026-06-30 12:00:08,2026-06-30 15:11:12


SyntaxError: invalid non-printable character U+EC7D (1080213637.py, line 3)

In [47]:
con.execute("""
CREATE OR REPLACE TEMP VIEW base_eventos_oper_chv AS
SELECT
    b.*,
    NULLIF(TRIM(CAST(t.NUM_OPER_CHV_INTRP AS VARCHAR)), '') AS NUM_OPER_CHV_INTRP
FROM base_eventos_exploratoria b
LEFT JOIN gold_interrupcao_tratada t
  ON TRIM(CAST(t.NUM_OCORRENCIA_ADMS AS VARCHAR)) = TRIM(CAST(b.NUM_OCORRENCIA_ADMS AS VARCHAR))
 AND TRIM(CAST(t.NUM_SEQ_INTRP AS VARCHAR)) = TRIM(CAST(b.NUM_SEQ_INTRP AS VARCHAR))
 AND TRIM(CAST(t.NUM_INTRP_UCI AS VARCHAR)) = TRIM(CAST(b.NUM_INTRP_UCI AS VARCHAR))
 AND TRIM(CAST(t.NUM_UC_UCI AS VARCHAR)) = TRIM(CAST(b.UC AS VARCHAR))
""")

con.execute("""
SELECT
    COUNT(*) AS LINHAS,
    SUM(CASE WHEN NUM_OPER_CHV_INTRP IS NOT NULL THEN 1 ELSE 0 END) AS LINHAS_COM_OPER_CHV,
    COUNT(DISTINCT NUM_OPER_CHV_INTRP) AS CHAVES_OPERACAO
FROM base_eventos_oper_chv
""").fetchdf()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,LINHAS,LINHAS_COM_OPER_CHV,CHAVES_OPERACAO
0,12262287,12262287.0,35757


In [ ]:
ocorrencias_mesma_chave_despacho_indevido = con.execute("""
WITH servicos_base AS (
    SELECT
        NULLIF(TRIM(CAST(PID_INTRP_SRVE AS VARCHAR)), '') AS NUM_SEQ_INTRP,
        NULLIF(TRIM(CAST(NUM_SEQ_SERV AS VARCHAR)), '') AS NUM_SEQ_SERV,
        NULLIF(TRIM(CAST(PID AS VARCHAR)), '') AS PID_SERVICO,
        LPAD(NULLIF(TRIM(CAST(COD_CAUSA_SRVE AS VARCHAR)), ''), 2, '0') AS COD_CAUSA_SRVE,
        CASE
            WHEN LPAD(NULLIF(TRIM(CAST(COD_CAUSA_SRVE AS VARCHAR)), ''), 2, '0') = '22'
            THEN 'ATENDIDO POR OUTRA OCORRENCIA'
            WHEN LPAD(NULLIF(TRIM(CAST(COD_CAUSA_SRVE AS VARCHAR)), ''), 2, '0') = '85'
            THEN 'IMPROCEDENTE'
            ELSE 'OUTRO'
        END AS DESC_CAUSA_SRVE,
        NULLIF(TRIM(CAST(COD_COMP_SRVE AS VARCHAR)), '') AS COD_COMP_SRVE,
        NULLIF(TRIM(CAST(COD_CONJTO_ELET_ANEEL AS VARCHAR)), '') AS CONJUNTO_SERVICO,
        NULLIF(TRIM(CAST(ESTADO_SERVICO_ACOMP AS VARCHAR)), '') AS ESTADO_SERVICO_ACOMP,
        NULLIF(TRIM(CAST(NUM_ORG_EXEC_SRV AS VARCHAR)), '') AS NUM_ORG_EXEC_SRV,
        DTHR_SOLIC_SRV,
        DTHR_GERA_SRV,
        DTHR_DESPACH_SRV,
        DTHR_SAIDA_SRV,
        DTHR_INIC_SRV,
        DTHR_TERM_SRV,
        DTHR_RETOR_SRV,
        DTHR_FECH_SRV,
        ARQUIVO_ORIGEM
    FROM backup_adms_servicos_junho
    WHERE NULLIF(TRIM(CAST(PID_INTRP_SRVE AS VARCHAR)), '') IS NOT NULL
),
eventos_uc AS (
    SELECT DISTINCT
        UC,
        CONJUNTO,
        REGIONAL,
        DATA_INTRP,
        NUM_OCORRENCIA_ADMS,
        NUM_SEQ_INTRP,
        NUM_INTRP_UCI,
        NUM_POSTO_UCI,
        NUM_OPER_CHV_INTRP,
        INICIO,
        FIM,
        TIPO_PROTOCOLO_UC,
        PROTOCOLO_RECUPERADO,
        COD_CAUSA_INTRP,
        COD_COMP_INTRP,
        CHI_LIQUIDO
    FROM base_eventos_oper_chv
    WHERE NULLIF(TRIM(CAST(NUM_OPER_CHV_INTRP AS VARCHAR)), '') IS NOT NULL
),
servicos_contexto AS (
    SELECT
        e.UC,
        e.CONJUNTO,
        e.REGIONAL,
        e.DATA_INTRP,
        e.NUM_OCORRENCIA_ADMS,
        e.NUM_SEQ_INTRP,
        e.NUM_INTRP_UCI,
        e.NUM_POSTO_UCI,
        e.NUM_OPER_CHV_INTRP,
        e.INICIO AS INICIO_INTERRUPCAO,
        e.FIM AS FIM_INTERRUPCAO,
        e.TIPO_PROTOCOLO_UC,
        e.PROTOCOLO_RECUPERADO,
        e.COD_CAUSA_INTRP,
        e.COD_COMP_INTRP,
        e.CHI_LIQUIDO,

        s.NUM_SEQ_SERV,
        s.PID_SERVICO,
        s.COD_CAUSA_SRVE,
        s.DESC_CAUSA_SRVE,
        s.COD_COMP_SRVE,
        s.ESTADO_SERVICO_ACOMP,
        s.NUM_ORG_EXEC_SRV,
        s.DTHR_SOLIC_SRV,
        s.DTHR_GERA_SRV,
        s.DTHR_DESPACH_SRV,
        s.DTHR_SAIDA_SRV,
        s.DTHR_INIC_SRV,
        s.DTHR_TERM_SRV,
        s.DTHR_RETOR_SRV,
        s.DTHR_FECH_SRV,
        s.ARQUIVO_ORIGEM
    FROM servicos_base s
    JOIN eventos_uc e
      ON TRIM(CAST(e.NUM_SEQ_INTRP AS VARCHAR)) = TRIM(CAST(s.NUM_SEQ_INTRP AS VARCHAR))
),
pares_mesma_chave AS (
    SELECT
        a.UC,
        a.CONJUNTO,
        a.REGIONAL,
        a.DATA_INTRP,
        a.NUM_OPER_CHV_INTRP,

        a.NUM_SEQ_SERV AS SERVICO_1,
        b.NUM_SEQ_SERV AS SERVICO_2,

        a.NUM_OCORRENCIA_ADMS AS OCORRENCIA_1,
        b.NUM_OCORRENCIA_ADMS AS OCORRENCIA_2,

        a.NUM_SEQ_INTRP AS INTERRUPCAO_1,
        b.NUM_SEQ_INTRP AS INTERRUPCAO_2,

        a.NUM_INTRP_UCI AS INTRP_UCI_1,
        b.NUM_INTRP_UCI AS INTRP_UCI_2,

        a.NUM_POSTO_UCI AS NUM_POSTO_UCI_1,
        b.NUM_POSTO_UCI AS NUM_POSTO_UCI_2,

        COALESCE(a.UC, '') || '-' ||
        COALESCE(a.NUM_SEQ_INTRP, '') || '-' ||
        COALESCE(a.NUM_INTRP_UCI, '') || '-' ||
        COALESCE(a.NUM_POSTO_UCI, '') || '-' ||
        COALESCE(a.NUM_OPER_CHV_INTRP, '') AS CHAVE_INTERRUPCAO_1,

        COALESCE(b.UC, '') || '-' ||
        COALESCE(b.NUM_SEQ_INTRP, '') || '-' ||
        COALESCE(b.NUM_INTRP_UCI, '') || '-' ||
        COALESCE(b.NUM_POSTO_UCI, '') || '-' ||
        COALESCE(b.NUM_OPER_CHV_INTRP, '') AS CHAVE_INTERRUPCAO_2,

        a.COD_CAUSA_SRVE AS COD_CAUSA_SERVICO_1,
        b.COD_CAUSA_SRVE AS COD_CAUSA_SERVICO_2,

        a.DESC_CAUSA_SRVE AS DESC_CAUSA_SERVICO_1,
        b.DESC_CAUSA_SRVE AS DESC_CAUSA_SERVICO_2,

        a.DTHR_INIC_SRV AS INICIO_SERVICO_1,
        b.DTHR_INIC_SRV AS INICIO_SERVICO_2,

        a.DTHR_FECH_SRV AS FECHAMENTO_SERVICO_1,
        b.DTHR_FECH_SRV AS FECHAMENTO_SERVICO_2,

        DATE_DIFF('minute', a.DTHR_FECH_SRV, b.DTHR_INIC_SRV) AS MIN_FECH_1_ATE_INIC_2,
        DATE_DIFF('minute', a.DTHR_FECH_SRV, b.DTHR_FECH_SRV) AS MIN_FECH_1_ATE_FECH_2,

        a.TIPO_PROTOCOLO_UC AS TIPO_PROTOCOLO_1,
        b.TIPO_PROTOCOLO_UC AS TIPO_PROTOCOLO_2,

        a.PROTOCOLO_RECUPERADO AS PROTOCOLO_1,
        b.PROTOCOLO_RECUPERADO AS PROTOCOLO_2,

        a.COD_CAUSA_INTRP AS COD_CAUSA_INTERRUPCAO_1,
        b.COD_CAUSA_INTRP AS COD_CAUSA_INTERRUPCAO_2,

        a.COD_COMP_INTRP AS COD_COMP_INTERRUPCAO_1,
        b.COD_COMP_INTRP AS COD_COMP_INTERRUPCAO_2,

        a.CHI_LIQUIDO AS CHI_LIQUIDO_1,
        b.CHI_LIQUIDO AS CHI_LIQUIDO_2,

        CASE
            WHEN a.DTHR_FECH_SRV IS NOT NULL
             AND b.DTHR_INIC_SRV IS NOT NULL
             AND DATE_DIFF('minute', a.DTHR_FECH_SRV, b.DTHR_INIC_SRV) BETWEEN 0 AND 10
            THEN 'S'
            ELSE 'N'
        END AS SEGUNDO_INICIA_ATE_10MIN,

        CASE
            WHEN a.DTHR_FECH_SRV IS NOT NULL
             AND b.DTHR_FECH_SRV IS NOT NULL
             AND DATE_DIFF('minute', a.DTHR_FECH_SRV, b.DTHR_FECH_SRV) BETWEEN 0 AND 10
            THEN 'S'
            ELSE 'N'
        END AS SEGUNDO_FECHA_ATE_10MIN,

        CASE
            WHEN b.COD_CAUSA_SRVE IN ('22', '85') THEN 'S'
            ELSE 'N'
        END AS SEGUNDO_SERVICO_22_85,

        CASE
            WHEN a.NUM_OCORRENCIA_ADMS <> b.NUM_OCORRENCIA_ADMS THEN 'S'
            ELSE 'N'
        END AS OMS_GEROU_OCORRENCIAS_DISTINTAS

    FROM servicos_contexto a
    JOIN servicos_contexto b
      ON a.UC = b.UC
     AND a.CONJUNTO = b.CONJUNTO
     AND a.NUM_OPER_CHV_INTRP = b.NUM_OPER_CHV_INTRP
     AND a.NUM_SEQ_SERV <> b.NUM_SEQ_SERV
     AND a.NUM_SEQ_INTRP <> b.NUM_SEQ_INTRP
     AND a.DTHR_FECH_SRV IS NOT NULL
     AND (
            b.DTHR_INIC_SRV IS NOT NULL
         OR b.DTHR_FECH_SRV IS NOT NULL
     )
     AND (
            DATE_DIFF('minute', a.DTHR_FECH_SRV, b.DTHR_INIC_SRV) BETWEEN 0 AND 10
         OR DATE_DIFF('minute', a.DTHR_FECH_SRV, b.DTHR_FECH_SRV) BETWEEN 0 AND 10
     )
)
SELECT *
FROM pares_mesma_chave
WHERE OMS_GEROU_OCORRENCIAS_DISTINTAS = 'S'
  AND (
        SEGUNDO_INICIA_ATE_10MIN = 'S'
     OR SEGUNDO_FECHA_ATE_10MIN = 'S'
  )
ORDER BY
    DATA_INTRP,
    UC,
    NUM_OPER_CHV_INTRP,
    FECHAMENTO_SERVICO_1,
    INICIO_SERVICO_2,
    FECHAMENTO_SERVICO_2
""").fetchdf()

ocorrencias_mesma_chave_despacho_indevido

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,UC,CONJUNTO,REGIONAL,DATA_INTRP,NUM_OPER_CHV_INTRP,SERVICO_1,SERVICO_2,OCORRENCIA_1,OCORRENCIA_2,INTERRUPCAO_1,...,COD_CAUSA_INTERRUPCAO_1,COD_CAUSA_INTERRUPCAO_2,COD_COMP_INTERRUPCAO_1,COD_COMP_INTERRUPCAO_2,CHI_LIQUIDO_1,CHI_LIQUIDO_2,SEGUNDO_INICIA_ATE_10MIN,SEGUNDO_FECHA_ATE_10MIN,SEGUNDO_SERVICO_22_85,OMS_GEROU_OCORRENCIAS_DISTINTAS
0,70196036,15581,LES,2026-06-24,82362C0826,20877853,20877927,1-26465471,1-26466521,261106332411063432,...,54,03,40,47,0.0,0.000000,S,N,N,S
1,100808670,16764,NRO,2026-06-25,8514818057,30496195,30496141,1-26466878,1-26465652,261106589011066026,...,01,39,22,62,1.7,1.911111,S,S,S,S
2,101423748,16764,NRO,2026-06-25,8514818057,30496195,30496141,1-26466878,1-26465652,261106589011066026,...,01,39,22,62,1.7,1.911111,S,S,S,S
3,101436092,16764,NRO,2026-06-25,8514818057,30496195,30496141,1-26466878,1-26465652,261106589011066026,...,01,39,22,62,1.7,1.911111,S,S,S,S
4,101530811,16764,NRO,2026-06-25,8514818057,30496195,30496141,1-26466878,1-26465652,261106589011066026,...,01,39,22,62,1.7,1.911111,S,S,S,S
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109,58374035,14678,CSL,2026-06-29,8669627175,10444123,10444970,1-26494988,1-26472223,261114085111141251,...,04,43,35,89,18.0,4.975278,S,N,N,S
110,60525355,14678,CSL,2026-06-29,8669627175,10444123,10444970,1-26494988,1-26472223,261114085111141251,...,04,43,35,89,18.0,4.975278,S,N,N,S
111,68074476,14678,CSL,2026-06-29,8669627175,10444123,10444970,1-26494988,1-26472223,261114085111141251,...,04,43,35,89,18.0,4.975278,S,N,N,S
112,83943773,14678,CSL,2026-06-29,8669627175,10444123,10444970,1-26494988,1-26472223,261114085111141251,...,04,43,35,89,18.0,4.975278,S,N,N,S


In [52]:
resumo_ocorrencias_mesma_chave = ocorrencias_mesma_chave_despacho_indevido.agg({
    "SERVICO_2": "nunique",
    "OCORRENCIA_2": "nunique",
    "UC": "nunique",
    "NUM_OPER_CHV_INTRP": "nunique",
}).rename({
    "SERVICO_2": "servicos_potencialmente_indevidos",
    "OCORRENCIA_2": "ocorrencias_potencialmente_duplicadas",
    "UC": "ucs_afetadas",
    "NUM_OPER_CHV_INTRP": "chaves_operacao_afetadas",
})

resumo_ocorrencias_mesma_chave

servicos_potencialmente_indevidos          3
ocorrencias_potencialmente_duplicadas      3
ucs_afetadas                             114
chaves_operacao_afetadas                   3
dtype: int64

In [53]:
resumo_ocorrencias_mesma_chave_sem_uc = con.execute("""
WITH servicos_base AS (
    SELECT
        NULLIF(TRIM(CAST(PID_INTRP_SRVE AS VARCHAR)), '') AS NUM_SEQ_INTRP,
        NULLIF(TRIM(CAST(NUM_SEQ_SERV AS VARCHAR)), '') AS NUM_SEQ_SERV,
        NULLIF(TRIM(CAST(PID AS VARCHAR)), '') AS PID_SERVICO,
        LPAD(NULLIF(TRIM(CAST(COD_CAUSA_SRVE AS VARCHAR)), ''), 2, '0') AS COD_CAUSA_SRVE,
        NULLIF(TRIM(CAST(COD_COMP_SRVE AS VARCHAR)), '') AS COD_COMP_SRVE,
        NULLIF(TRIM(CAST(COD_CONJTO_ELET_ANEEL AS VARCHAR)), '') AS CONJUNTO_SERVICO,
        NULLIF(TRIM(CAST(ESTADO_SERVICO_ACOMP AS VARCHAR)), '') AS ESTADO_SERVICO_ACOMP,
        DTHR_SOLIC_SRV,
        DTHR_GERA_SRV,
        DTHR_DESPACH_SRV,
        DTHR_INIC_SRV,
        DTHR_TERM_SRV,
        DTHR_FECH_SRV
    FROM backup_adms_servicos_junho
    WHERE NULLIF(TRIM(CAST(PID_INTRP_SRVE AS VARCHAR)), '') IS NOT NULL
),
eventos_interrupcao AS (
    SELECT DISTINCT
        CONJUNTO,
        REGIONAL,
        DATA_INTRP,
        NUM_OCORRENCIA_ADMS,
        NUM_SEQ_INTRP,
        NUM_OPER_CHV_INTRP,
        INICIO,
        FIM,
        TIPO_PROTOCOLO_UC,
        PROTOCOLO_RECUPERADO,
        COD_CAUSA_INTRP,
        COD_COMP_INTRP
    FROM base_eventos_oper_chv
    WHERE NULLIF(TRIM(CAST(NUM_OPER_CHV_INTRP AS VARCHAR)), '') IS NOT NULL
),
servicos_contexto AS (
    SELECT DISTINCT
        e.CONJUNTO,
        e.REGIONAL,
        e.DATA_INTRP,
        e.NUM_OCORRENCIA_ADMS,
        e.NUM_SEQ_INTRP,
        e.NUM_OPER_CHV_INTRP,
        e.INICIO AS INICIO_INTERRUPCAO,
        e.FIM AS FIM_INTERRUPCAO,
        e.TIPO_PROTOCOLO_UC,
        e.PROTOCOLO_RECUPERADO,
        e.COD_CAUSA_INTRP,
        e.COD_COMP_INTRP,

        s.NUM_SEQ_SERV,
        s.PID_SERVICO,
        s.COD_CAUSA_SRVE,
        s.COD_COMP_SRVE,
        s.ESTADO_SERVICO_ACOMP,
        s.DTHR_SOLIC_SRV,
        s.DTHR_GERA_SRV,
        s.DTHR_DESPACH_SRV,
        s.DTHR_INIC_SRV,
        s.DTHR_TERM_SRV,
        s.DTHR_FECH_SRV
    FROM servicos_base s
    JOIN eventos_interrupcao e
      ON TRIM(CAST(e.NUM_SEQ_INTRP AS VARCHAR)) = TRIM(CAST(s.NUM_SEQ_INTRP AS VARCHAR))
),
pares_mesma_chave AS (
    SELECT
        a.CONJUNTO,
        a.REGIONAL,
        a.DATA_INTRP,
        a.NUM_OPER_CHV_INTRP,

        a.NUM_SEQ_SERV AS SERVICO_1,
        b.NUM_SEQ_SERV AS SERVICO_2,

        a.NUM_OCORRENCIA_ADMS AS OCORRENCIA_1,
        b.NUM_OCORRENCIA_ADMS AS OCORRENCIA_2,

        a.NUM_SEQ_INTRP AS INTERRUPCAO_1,
        b.NUM_SEQ_INTRP AS INTERRUPCAO_2,

        a.COD_CAUSA_SRVE AS COD_CAUSA_SERVICO_1,
        b.COD_CAUSA_SRVE AS COD_CAUSA_SERVICO_2,

        a.COD_COMP_SRVE AS COD_COMP_SERVICO_1,
        b.COD_COMP_SRVE AS COD_COMP_SERVICO_2,

        a.COD_CAUSA_INTRP AS COD_CAUSA_INTERRUPCAO_1,
        b.COD_CAUSA_INTRP AS COD_CAUSA_INTERRUPCAO_2,

        a.COD_COMP_INTRP AS COD_COMP_INTERRUPCAO_1,
        b.COD_COMP_INTRP AS COD_COMP_INTERRUPCAO_2,

        a.TIPO_PROTOCOLO_UC AS TIPO_PROTOCOLO_1,
        b.TIPO_PROTOCOLO_UC AS TIPO_PROTOCOLO_2,

        a.PROTOCOLO_RECUPERADO AS PROTOCOLO_1,
        b.PROTOCOLO_RECUPERADO AS PROTOCOLO_2,

        a.DTHR_SOLIC_SRV AS SOLICITACAO_SERVICO_1,
        b.DTHR_SOLIC_SRV AS SOLICITACAO_SERVICO_2,

        a.DTHR_DESPACH_SRV AS DESPACHO_SERVICO_1,
        b.DTHR_DESPACH_SRV AS DESPACHO_SERVICO_2,

        a.DTHR_INIC_SRV AS INICIO_SERVICO_1,
        b.DTHR_INIC_SRV AS INICIO_SERVICO_2,

        a.DTHR_TERM_SRV AS TERMINO_SERVICO_1,
        b.DTHR_TERM_SRV AS TERMINO_SERVICO_2,

        a.DTHR_FECH_SRV AS FECHAMENTO_SERVICO_1,
        b.DTHR_FECH_SRV AS FECHAMENTO_SERVICO_2,

        DATE_DIFF('minute', a.DTHR_FECH_SRV, b.DTHR_INIC_SRV) AS MIN_FECH_1_ATE_INIC_2,
        DATE_DIFF('minute', a.DTHR_FECH_SRV, b.DTHR_FECH_SRV) AS MIN_FECH_1_ATE_FECH_2

    FROM servicos_contexto a
    JOIN servicos_contexto b
      ON a.CONJUNTO = b.CONJUNTO
     AND a.NUM_OPER_CHV_INTRP = b.NUM_OPER_CHV_INTRP
     AND a.NUM_SEQ_SERV <> b.NUM_SEQ_SERV
     AND a.NUM_SEQ_INTRP <> b.NUM_SEQ_INTRP
     AND a.NUM_OCORRENCIA_ADMS <> b.NUM_OCORRENCIA_ADMS
     AND a.DTHR_FECH_SRV IS NOT NULL
     AND (
            DATE_DIFF('minute', a.DTHR_FECH_SRV, b.DTHR_INIC_SRV) BETWEEN 0 AND 10
         OR DATE_DIFF('minute', a.DTHR_FECH_SRV, b.DTHR_FECH_SRV) BETWEEN 0 AND 10
     )
)
SELECT DISTINCT *
FROM pares_mesma_chave
ORDER BY
    DATA_INTRP,
    CONJUNTO,
    NUM_OPER_CHV_INTRP,
    FECHAMENTO_SERVICO_1,
    INICIO_SERVICO_2,
    FECHAMENTO_SERVICO_2
""").fetchdf()

resumo_ocorrencias_mesma_chave_sem_uc

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,CONJUNTO,REGIONAL,DATA_INTRP,NUM_OPER_CHV_INTRP,SERVICO_1,SERVICO_2,OCORRENCIA_1,OCORRENCIA_2,INTERRUPCAO_1,INTERRUPCAO_2,...,DESPACHO_SERVICO_1,DESPACHO_SERVICO_2,INICIO_SERVICO_1,INICIO_SERVICO_2,TERMINO_SERVICO_1,TERMINO_SERVICO_2,FECHAMENTO_SERVICO_1,FECHAMENTO_SERVICO_2,MIN_FECH_1_ATE_INIC_2,MIN_FECH_1_ATE_FECH_2
0,15581,LES,2026-06-24,82362C0826,20877853,20877927,1-26465471,1-26466521,261106332411063432,261106938211070109,...,2026-06-24 22:36:00,2026-06-25 11:01:00,2026-06-25 12:27:00,2026-06-25 13:22:00,2026-06-25 13:20:00,2026-06-25 14:22:00,2026-06-25 13:20:00,2026-06-25 14:22:00,2,62
1,16764,NRO,2026-06-25,8514818057,30496195,30496141,1-26466878,1-26465652,261106589011066026,261106430511064310,...,2026-06-25 06:50:00,2026-06-25 06:50:00,2026-06-25 07:41:00,2026-06-25 08:08:00,2026-06-25 08:06:00,2026-06-25 08:13:00,2026-06-25 08:06:00,2026-06-25 08:13:00,2,7
2,14640,OES,2026-06-26,81776V5437,50890102,50890343,1-26471608,1-26472397,261107970911080209,261108032311084412,...,2026-06-26 08:14:00,2026-06-26 11:22:00,2026-06-26 11:14:00,2026-06-26 12:03:00,2026-06-26 11:56:00,2026-06-26 12:08:00,2026-06-26 11:56:00,2026-06-26 12:08:00,7,12
3,16760,OES,2026-06-27,81776BRY18,50891166,50894150,1-26476099,1-26483251,261110427311104879,261110492711105689,...,2026-06-27 09:00:00,2026-06-28 09:57:00,2026-06-28 09:24:00,2026-06-28 10:15:00,2026-06-28 10:12:00,2026-06-28 11:11:00,2026-06-28 10:12:00,2026-06-28 11:11:00,3,59
4,16760,OES,2026-06-27,81776BRY18,50890896,50891023,1-26480855,1-26480749,261110579311105927,261110603911106051,...,2026-06-27 08:06:00,2026-06-27 08:09:00,2026-06-28 11:17:00,2026-06-28 11:36:00,2026-06-28 11:28:00,2026-06-28 11:38:00,2026-06-28 11:28:00,2026-06-28 11:38:00,8,10
5,14628,OES,2026-06-28,83240G0270,50894286,50891956,1-26483579,1-26477962,261111913211119133,261111910911119110,...,2026-06-28 11:07:00,2026-06-27 13:49:00,2026-06-28 16:56:00,2026-06-28 17:23:00,2026-06-28 17:22:00,2026-06-28 17:29:00,2026-06-28 17:22:00,2026-06-28 17:29:00,1,7
6,16760,OES,2026-06-28,81776BRY18,50894150,50890896,1-26483251,1-26480855,261110492711105689,261110579311105927,...,2026-06-28 09:57:00,2026-06-27 08:06:00,2026-06-28 10:15:00,2026-06-28 11:17:00,2026-06-28 11:11:00,2026-06-28 11:28:00,2026-06-28 11:11:00,2026-06-28 11:28:00,6,17
7,14678,CSL,2026-06-29,8669627175,10444123,10444970,1-26494988,1-26472223,261114085111141251,261114266611147533,...,2026-06-30 07:15:00,2026-06-30 08:17:00,2026-06-30 09:13:00,2026-06-30 09:44:00,2026-06-30 09:41:00,2026-06-30 10:42:00,2026-06-30 09:41:00,2026-06-30 10:42:00,3,61
8,14678,CSL,2026-06-29,86696C0233,10443183,10443261,1-26489829,1-26490185,261111942111120108,261112014611120531,...,2026-06-29 08:01:00,2026-06-29 08:31:00,2026-06-29 08:15:00,2026-06-29 08:46:00,2026-06-29 08:44:00,2026-06-29 09:04:00,2026-06-29 08:44:00,2026-06-29 09:04:00,2,20
9,16760,OES,2026-06-29,81776BRY18,50897402,50896869,1-26498235,1-26495365,261113654311136736,261117840011179497,...,2026-06-29 20:08:00,2026-06-29 17:46:00,2026-06-29 21:50:00,2026-06-29 22:13:00,2026-06-29 22:08:00,2026-06-29 22:41:00,2026-06-29 22:08:00,2026-06-29 22:41:00,5,33
